In [ ]:
################################## Test the bbknn result based on endothelial lineage
import pickle 
import os
import re
import pandas as pd
import numpy as np
import sys
import scanpy as sc
# import scatlasvae  ## 
import scipy.sparse as sp
import scipy
import bbknn
import harmonypy as hm
import gc
import weakref
import resource

import sys, os
print("=== 当前 Jupyter 使用的信息 ===")
print("Python 路径：", sys.executable)
print("Python 版本：", sys.version.split()[0])
print("Conda 环境：", os.environ.get('CONDA_DEFAULT_ENV'))
import socket
print(socket.gethostname())
!hostname
import os, platform, socket
print("Node 主机名:", socket.gethostname())
print("系统版本:", platform.platform())
print("CPU 核心数:", os.cpu_count())
print("当前工作目录:", os.getcwd())
os.chdir('/data/project/chenfeiwang')


In [ ]:
cell_lineage_dict = {
    "T": [
        "CD3D", "CD3E", "CD3G",  # T细胞核心标记
        "CD4", "CD8A", "CD8B",    # CD4+/CD8+ T亚群
        "FOXP3", "IL2RA",          # Treg细胞标记
        "TRAC", "TRBC1", "TRBC2"  # TCR基因
    ],
    "B": [
        "CD19", "CD79A", "CD79B",  # B细胞核心标记
        "MS4A1",                   # CD20
        "IGHD", "IGHM",            # B细胞受体
        "CD27", "CD38"             # 记忆B/浆细胞标记
    ],
    "Fibro": [
        "COL1A1", "COL1A2",        # 胶原蛋白
        "COL3A1", "COL6A1", 
        "DCN", "LUM",              # 基质蛋白
        "FAP", "PDPN",             # 成纤维活化标记
        "ACTA2"                    # 肌成纤维标记（α-SMA）
    ],
    "Endo": [
        "PECAM1", "CDH5",         # 内皮粘附分子
        "VWF", "CLDN5",           # Weibel-Palade小体标记
        "SELE", "ICAM1",           # 活化内皮标记
        "FLT1", "KDR"              # VEGF受体
    ],
    "MoMac": [
        "CD14", "CD68",            # 单核/巨噬核心标记
        "FCGR3A", "CD16",          # 非经典单核
        "LYZ", "CST3",             # 溶酶体相关
        "S100A8", "S100A9",       # 钙结合蛋白
        "ITGAM"                    # CD11b
    ],
    "Epi": [
        "EPCAM", "CDH1",           # 上皮粘附分子
        "KRT8", "KRT18",           # 角蛋白
        "MUC1", "TFF3",            # 分泌蛋白
        "SFN", "CLDN3"             # 紧密连接蛋白
    ]
}


In [ ]:
gene_marker_list = [gene for gene in cell_lineage_dict['T'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='T')
gene_marker_list = [gene for gene in cell_lineage_dict['B'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='B')
gene_marker_list = [gene for gene in cell_lineage_dict['Fibro'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Fibro')
gene_marker_list = [gene for gene in cell_lineage_dict['Endo'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Endo')
gene_marker_list = [gene for gene in cell_lineage_dict['MoMac'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='MoMac')
gene_marker_list = [gene for gene in cell_lineage_dict['Epi'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Epi')   


In [ ]:
########### Calculate the 6 Lineage celltype scores  & Fib signature scores
Fibro_marker_sub_ya =  {
    "ECM": ["ACTA2","TAGLN","BGN","COL8A1","COL15A1","IGFBP7","TPM1","TPM2","COL10A1","POSTN","MYL9","COL13A1","COL14A1","MYH11","MYLK","ACTG2","FN1",
"LUM","DCN","VCAN", "COL5A1" , "COL5A2" ,"COL63A" ,"PDGFA" ,"MMP2" , "COL1A2",  "RGS5"], # Fibroblasts
    "Immune": ["C5", "C5AR1", "TGFB1", "TGFBR1", "TGFB2" ,"TGFBR1", "CXCL12", "CXCR4","CXCL13", "CXCR5", "C7", "CFB","CFH", "CFI",
"BCAM","F11R","IRF5","CCL2", "IL6", "CXCL2", "CXCL1","CXCL3"], # Immune regulatory 
    "AP": ["CD74", "HLA-DQA1",  "CD83", "HLA-DRA", "HLA-DPA1" , "HLA-DQA", "HLA-DRB1", "H2-Q4"] }

Fibro_marker_sub_ya_inte = []
for key, genes in Fibro_marker_sub_ya.items():
    valid_genes = [gene for gene in genes if gene in adata.var_names]
    Fibro_marker_sub_ya_inte.extend([(gene, key) for gene in valid_genes])

# 计算signature score
gene_marker_list = [gene for gene in Fibro_marker_sub_ya['ECM'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='ECM')

gene_marker_list = [gene for gene in Fibro_marker_sub_ya['Immune'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Immune')

gene_marker_list = [gene for gene in Fibro_marker_sub_ya['AP'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='AP')


In [ ]:
########### Then calculate the Fibroblast subtype marker list scores
fibroblast_subtype_markers = {
  'Myfibro': [
    'ACTA2', 'TAGLN', 'TPM1', 'CALD1'  # 最特异的肌成纤维细胞marker, PDGFRB 改 CALD1,PDGFRB 表达很低; 另TPM1 > TPM2> MYL9, MYL9 改掉
  ],
  #  'apFibro': [
  #      'CD74', 'HLA-DRA', 'HLA-DRB1', 'CIITA'  # 抗原呈递相关核心基因
  #  ],
  'eFibro': [
    'COL1A1', 'COL3A1', 'IGFBP7', 'DCN'  # ECM核心成分, LUM 替换成 IGFBP7, adata中无 LUM
  ],
  'iFibro': [
    'HLA-DRA', 'HLA-DRB1','CD74', 'ARHGDIB' # 免疫调节关键因子,加'ARHGDIB', 另'CCL2','CXCL12','ICAM1'这几个gene表达都很低,去掉,甚至 fibroblast_subtype_markers_full 中都可以去掉
  ],
  'qFibro': [
    'CLU', 'APOD', 'CFD', 'MGP'  # 静息状态特征基因, adata中无 PI16, 换成 MGP
  ]
}
fibroblast_subtype_markers_full = {
  'Myfibro1': [
    'ACTA2', 'TAGLN','TPM1' ,'CALD1',
    'PDGFRB','CNN1', 'ACTG2','MYLK','DES','MYL9','TUBB2B','TPM2'    # added
  ],
  'eFibro1': [
    'COL1A1', 'COL3A1', 'IGFBP7', 'DCN', 
    'COL6A1', 'COL6A2', 'COL6A3','FN1',  'BGN', 'COL1A2', 'SPARC', 'POSTN', 'TNC', 'VCAN'     # added
  ],
  'iFibro1': [
    'HLA-DRA', 'HLA-DRB1','CD74', 'ARHGDIB' ,  
    'IL6',  'CIITA','CCL2','CXCL12',  'ICAM1' , 'CCL19', 'CCL21', 'CXCL14','IL11', 'PTGS2'  # added
  ],
  'qFibro1': [
    'CLU', 'APOD', 'CFD', 'MGP', 
    'CRABP1', 'FBLN1','FBLN2', 'MFAP4', 'GSN', 'SPARCL1',  'IGFBP6'   # added
  ]
}



In [ ]:
# 计算 fibroblast_subtype_markers 的 signature
gene_marker_list = [gene for gene in fibroblast_subtype_markers['Myfibro'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Myfibro')
# gene_marker_list = [gene for gene in fibroblast_subtype_markers['apFibro'] if gene in adata.var_names]
# sc.tl.score_genes(adata, gene_marker_list, score_name='apFibro')
gene_marker_list = [gene for gene in fibroblast_subtype_markers['eFibro'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='eFibro')
gene_marker_list = [gene for gene in fibroblast_subtype_markers['iFibro'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='iFibro')
gene_marker_list = [gene for gene in fibroblast_subtype_markers['qFibro'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='qFibro')
# 计算fibroblast_subtype_markers_full 的 signature score
gene_marker_list = [gene for gene in fibroblast_subtype_markers_full['Myfibro1'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Myfibro1')
# gene_marker_list = [gene for gene in fibroblast_subtype_markers_full['apFibro1'] if gene in adata.var_names]
# sc.tl.score_genes(adata, gene_marker_list, score_name='apFibro1')
gene_marker_list = [gene for gene in fibroblast_subtype_markers_full['eFibro1'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='eFibro1')
gene_marker_list = [gene for gene in fibroblast_subtype_markers_full['iFibro1'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='iFibro1')
gene_marker_list = [gene for gene in fibroblast_subtype_markers_full['qFibro1'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='qFibro1')



In [ ]:
import scanpy as sc
import numpy as np

def calculate_simple_scores(adata, marker_dict, score_prefix=''):
    """
    计算简单的平均表达得分（绝对非负）
    """
    for key, genes in marker_dict.items():
        score_name = f"{score_prefix}{key}" if score_prefix else key
        
        # 过滤存在的基因
        valid_genes = [g for g in genes if g in adata.var_names]
        
        if len(valid_genes) == 0:
            print(f"Warning: No genes found for {key}")
            adata.obs[score_name] = 0
            continue
        
        print(f"Calculating {score_name} with {len(valid_genes)} genes...")
        # 计算平均表达
        # 注意：对于log-normalized数据，可以直接平均
        expr_data = adata[:, valid_genes].X
        if scipy.sparse.issparse(expr_data):
            scores = expr_data.mean(axis=1).A1  # A1 converts to 1D array
        else:
            scores = expr_data.mean(axis=1)
        
        # 确保非负
        scores = np.maximum(scores, 0)
        adata.obs[score_name] = scores
        print(f"  Range: [{scores.min():.4f}, {scores.max():.4f}]")

# 计算所有得分
print("Calculating all fibroblast subtype scores...")
calculate_simple_scores(adata, fibroblast_subtype_markers)
calculate_simple_scores(adata, fibroblast_subtype_markers_full)
calculate_simple_scores(adata,Fibro_marker_sub_ya )
calculate_simple_scores(adata,cell_lineage_dict ) 
calculate_simple_scores(adata,stromal_marker ) 



# 验证leiden=8的Myfibro得分
print("\n" + "="*60)
print("Verification for leiden=8:")
print("="*60)

# 检查leiden=8的细胞
leiden8_cells = adata.obs['leiden'] == '8'

if any(leiden8_cells):
    myfibro_scores = adata.obs.loc[leiden8_cells, 'Myfibro']
    print(f"Myfibro score for leiden=8 cells:")
    print(f"  Mean: {myfibro_scores.mean():.4f}")
    print(f"  Min: {myfibro_scores.min():.4f}")
    print(f"  Max: {myfibro_scores.max():.4f}")
    print(f"  All scores >= 0: {(myfibro_scores >= 0).all()}")
    
    # 检查各个基因的表达
    myfibro_genes = [g for g in fibroblast_subtype_markers['Myfibro'] if g in adata.var_names]
    if myfibro_genes:
        print(f"\nIndividual gene expression in leiden=8 cells:")
        for gene in myfibro_genes:
            gene_expr = adata[leiden8_cells, gene].X
            if scipy.sparse.issparse(gene_expr):
                gene_expr = gene_expr.toarray().flatten()
            mean_expr = np.mean(gene_expr)
            print(f"  {gene}: {mean_expr:.4f}")
else:
    print("No cells found for leiden=8")

In [ ]:
## 重新再算一下
Fib_marker_deg = ['DCN', 'COL1A2', 'LUM', 'MGP', 'COL1A1', 'COL3A1', 'CALD1', 'IGFBP7', 'COL6A2', 'SPARC']
Endo_marker_deg =['VWF', 'PECAM1', 'TM4SF1', 'RAMP2', 'GNG11', 'CDH5', 'TEK', 'KDR']  
stromal_marker = {
    'Fib_deg': Fib_marker_deg,
    'Endo_deg': Endo_marker_deg
}
gene_marker_list = [gene for gene in Fib_marker_deg if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Fib_deg')
gene_marker_list = [gene for gene in Endo_marker_deg if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Endo_deg') 


In [ ]:
########## 画 6Linage的code
genesets = ['T','B','Fibro', 'Endo','MoMac', 'Epi']
global_min = min([adata.obs[gs].min() for gs in genesets if gs in adata.obs])
global_max = max([adata.obs[gs].max() for gs in genesets if gs in adata.obs])
print(f"统一颜色范围: [{global_min:.3f}, {global_max:.3f}]")
# 绘制UMAP图，使用统一的vmin和vmax
axes = sc.pl.umap(adata, 
                  color=genesets,
                  color_map='viridis',
                  title=['T','B','Fibro', 'Endo','MoMac', 'Epi'],
                  frameon=False,
                  ncols=3,
                  size=3,
                  vmin=global_min,  # 统一最小值
                  vmax=global_max,  # 统一最大值
                  show=False,
                  legend_fontsize=18,
                  legend_fontoutline=2,
                  wspace=0.1)
for i, ax in enumerate(axes):
    ax.set_title(f'{ax.get_title()}', fontsize=20)  #   fontweight='bold'
    

# 调整图形大小
plt.gcf().set_size_inches(18, 9)
fig = plt.gcf()
# 创建一个用于colorbar的mappable对象
import matplotlib.cm as cm
from matplotlib.colors import Normalize
norm = Normalize(vmin=global_min, vmax=global_max)
sm = cm.ScalarMappable(norm=norm, cmap='viridis')
sm.set_array([])
plt.tight_layout()
fig_path = 'external_inputs/umap'
# plt.savefig(
#     fig_path+"6Lineage_umap.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )  ## pdf 存的好大，后续如果需要PDF文件需要优化下

plt.savefig(fig_path+"/6Lineage_umap.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False)
plt.show()
plt.close()


In [ ]:
########## 画 Fibro中 signature list 的UMAP图
genesets = ['ECM','Immune','AP', 'leiden']
axes = sc.pl.umap(adata, 
                  color=genesets,
                  color_map='viridis',
                  title=['ECM','Immune','AP', 'leiden'],
                  frameon=False,
                  ncols=2,
                  size=3,
                  vmin=global_min,  # 统一最小值(6Lineage那里算了)
                  vmax=global_max,  # 统一最大值
                  show=False,
                  legend_fontsize=18,
                  legend_fontoutline=2,
                  wspace=0.1)
for i, ax in enumerate(axes):
    ax.set_title(f'{ax.get_title()}', fontsize=20)  #   fontweight='bold'

    
plt.gcf().set_size_inches(11, 9)
fig = plt.gcf()
plt.tight_layout()
fig_path = 'external_inputs/umap'
# plt.savefig(
#     fig_path+"Fib_all_umap.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_path+"Fib_all_umap.png", dpi=300, bbox_inches='tight') 
plt.show()


In [ ]:
##### 画 Endo 中 subtype的code
genesets = ['Myfibro', 'eFibro', 'iFibro', 'qFibro', 'Myfibro1', 'eFibro1', 'iFibro1', 'qFibro1']
axes = sc.pl.umap(adata, 
                  color=genesets,
                  color_map='viridis',
                  title=genesets,
                  frameon=False,
                  ncols=3,
                  size=3,
                  vmin=global_min,  # 统一最小值(6Lineage那里算了)
                  vmax=global_max,  # 统一最大值
                  show=False,
                  legend_fontsize=18,
                  legend_fontoutline=2,
                  wspace=0.1)
for i, ax in enumerate(axes):
    ax.set_title(f'{ax.get_title()}', fontsize=20)  #   fontweight='bold'


plt.gcf().set_size_inches(18, 12)
fig = plt.gcf()
plt.tight_layout()
fig_path = 'external_inputs/umap'
# plt.savefig(
#     fig_path+"Fibro_all_umap.pdf",
#     dpi=150,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_path+"Fib_subtype_umap.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False )
plt.show()



In [ ]:
##### 画 Leiden 和 其他 tissue 这些的代码 
plt.rcParams['figure.figsize'] = (6, 5)
fig, ax = plt.subplots()
# 关键：指定 palette=Nat_color_list
sc.pl.umap(
    adata,
    color='leiden',
    # palette=Nat_color_list,
    ax=ax,
    show=False,
    legend_loc='on data',   # 不用scanpy默认legend
    s=10,
    alpha=0.8
)
ax.set_title('Leiden', fontsize=16, fontweight='bold', pad=20)
# ================================
# 6. 去边框
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
fig_dir='external_inputs/umap'
# plt.savefig(
#     fig_dir+"Fibro_leiden.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_dir+"Fib_leiden.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False )
plt.show()


In [ ]:
##### 画 tissue 的代码 
plt.rcParams['figure.figsize'] = (10, 5)
fig, ax = plt.subplots()
# 关键：指定 palette=Nat_color_list
sc.pl.umap(
    adata,
    color='tissue',
    # palette=Nat_color_list,
    ax=ax,
    show=False,
    s=10,
    alpha=0.8
)
ax.set_title('Tissue', fontsize=16, fontweight='bold', pad=20)
# ================================
# 6. 去边框
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
fig_dir='external_inputs/umap'
# plt.savefig(
#     fig_dir+"Fibro_leiden.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_dir+"Fib_tissue.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False )
plt.show()


In [ ]:
######## 画每个leiden 中有多少细胞柱状图的code 
import matplotlib.pyplot as plt
import pandas as pd
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']  # 解决中文显示（可选）
plt.rcParams['axes.unicode_minus'] = False

# 创建画布
fig, ax = plt.subplots(figsize=(16, 8))  # 画布大小可根据你的簇数调整
# 绘制柱状图
bars = ax.bar(
    x=df_sorted['leiden'],
    height=df_sorted['cell_count'],
    color='#1f77b4',  # 专业配色，可替换为你喜欢的颜色
    edgecolor='black',
    linewidth=0.8
)

# ✅ 关键：对数刻度y轴，完美解决大小簇差距过大
ax.set_yscale('log')
ax.set_ylim(bottom=df_sorted['cell_count'].min() * 0.8, top=df_sorted['cell_count'].max() * 1.1)

# ✅ 柱子顶部标注细胞数（千分位分隔，垂直显示避免拥挤）
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height * 1.02,  # 对数刻度下用比例偏移更合理
        f'{int(height):,}',  # 千分位分隔，如124,381
        ha='center',
        va='bottom',
        fontsize=8,
        rotation=90
    )

# ✅ 图表美化与标签
ax.set_title('Leiden CellNum(Log Scale)', fontsize=18, pad=20)
ax.set_xlabel('Leiden', fontsize=14, labelpad=15)
ax.set_ylabel('CellNum(Log Scale)', fontsize=14, labelpad=15)
ax.tick_params(axis='x', rotation=90, labelsize=10)  # x轴标签旋转，避免重叠
ax.tick_params(axis='y', labelsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.7)  # 横向网格线，方便读数
# 调整布局，避免标签截断
plt.tight_layout()
# ✅ 保存高清图片（300dpi满足顶刊要求）
plt.savefig(
    'external_inputs/leiden_cell_count_log.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()


In [ ]:
Fib_marker_general = sorted(list(set(["COL1A1","COL1A2","COL3A1","COL6A1","IL6","MCAM","MMP2","MYL9", "MYLK","PDGFA"])))
Fib_marker_spec =sorted(list(set(['COL1A1','CFD','CTHRC1','POSTN','FAP','LRRC15','HAND2','SFRP1','IL6','CXCL3','APOD','PI16',
  'GPX1','IGF2','CCL19','SAA1','VWA1','CCL5','CCL4','CILP','THBS4','MYH11','MYLK','GJA4','ACTA2','RGS5'])))  # 'PTPRC'
Fib_marker_yilv = ['IGF2','ENG','FAP','PDGFRA','PECAM1','ITGB1','SPON1','MMP2','MYO1B','ARHGDIB','NDUFA4L2','HIGD1B','COX4I2','RGS5',
    'CNN1','DES','FGFR4','VEGFD','MME','LEFR','PI16','C3','CFD','C7','ADH1B','GREM1','CTHRC1','LRRC15','POSTN','MYH11','ACTA2','PDPN',
    'CXCL14','CXCL12','CCL2','COL12A1','COL11A1','COL10A1','COL8A1','COL5A1','COL4A2','COL4A1','COL3A1','COL1A2','COL1A1'] # ,'CD34'
Fib_marker_union = list(set(Fib_marker_general) | set(Fib_marker_spec) | set(Fib_marker_yilv))
Fib_marker_deg = ['DCN', 'COL1A2', 'LUM', 'MGP', 'COL1A1', 'COL3A1', 'CALD1', 'IGFBP7', 'COL6A2', 'SPARC']

iFibro_extended_markers = ['CXCL12', 'IL6', 'CCL2', 'ICAM1', 'CXCL1', 'CXCL8', 'IL33', 'PTGS2']
confusion_markers = {
    'apFibro': ['CD74', 'HLA-DRA', 'CIITA'],
    'eFibro': ['COL1A1', 'COL3A1'],
    'Myfibro': ['ACTA2', 'TAGLN'],
    'qFibro': ['CLU', 'APOD']
}
all_markers = iFibro_extended_markers + [m for lst in confusion_markers.values() for m in lst]
genes_union = set().union(*fibroblast_subtype_markers.values(), *fibroblast_subtype_markers_full.values())
Fib_marker_all = list(set(Fib_marker_union) | set(Fib_marker_deg) | set(all_markers) | set(genes_union) | set(['CALD1','TUBB2B','CDKN2A','SPARC','QKI']))


In [ ]:
# 定义要保留的基因   
gene_keep = [
  'QKI','APOD', 'CFD', 'CLU','PI16', # qFibro
    'CCL2', 'ARHGDIB',  'IL33', 'CD74',  # iFibro,    'APOD','PI16', 'CXCL14','B2M',
  'SPARC','COL1A1', 'COL1A2', 'COL6A3','LUM',  'COL3A1','CDKN2A', # eFibro ; DCN 去掉，在很多celltype都高表达(特别qFibro)
  "MYH11",'MYL9','TPM2','TAGLN','CALD1','TUBB2B' # Myfibro, 原来的   "ACTA2",'GJA4','RGS5'
]   


In [ ]:
######## 为了更好的看 Fib_marker_all 中的基因在每个 Leiden中的top表达基因，因为看单个基因的UMAP肉眼容易出错，下面进行定量查看,另外加上 leiden, Celltype, 以及对基因进行 group 等操作
import numpy as np
import pandas as pd
from scipy.sparse import issparse

### Fib_marker_all & fibroblast_subtype_markers_full 上面都有, 另外 fibroblast_subtype_markers_full 中必有 fibroblast_subtype_markers的基因

# 1. 创建基因分类映射
gene_to_subtype = {}
for subtype, genes in fibroblast_subtype_markers_full.items():
    for gene in genes:
        if gene in Fib_marker_all:
            gene_to_subtype[gene] = subtype

# 为不在任何subtype中的基因标记为'Other'
for gene in Fib_marker_all:
    if gene not in gene_to_subtype:
        gene_to_subtype[gene] = 'Other'

# 2. 定义subtype的顺序
subtype_order = ['iFibro1', 'eFibro1', 'Myfibro1', 'qFibro1', 'Other']

# 3. 对基因进行排序
genes_by_subtype = {subtype: [] for subtype in subtype_order}
for gene in Fib_marker_all:
    subtype = gene_to_subtype[gene]
    genes_by_subtype[subtype].append(gene)

# 按指定顺序拼接基因列表
sorted_genes = []
for subtype in subtype_order:
    sorted_genes.extend(genes_by_subtype[subtype])

print(f"Sorted {len(sorted_genes)} genes by subtype")

# 4. 从adata中高效提取表达数据
print("Extracting expression data...")

# 确保所有基因都在adata中
existing_genes = [gene for gene in sorted_genes if gene in adata.var_names]
missing_genes = set(sorted_genes) - set(existing_genes)
if missing_genes:
    print(f"Warning: {len(missing_genes)} genes not found in adata: {list(missing_genes)[:5]}...")

# 高效提取表达矩阵
if issparse(adata.X):
    expression_matrix = adata[:, existing_genes].X.toarray()
else:
    expression_matrix = adata[:, existing_genes].X

# 创建DataFrame
expression_df = pd.DataFrame(
    expression_matrix,
    index=adata.obs_names,
    columns=existing_genes
)

# 添加leiden信息
expression_df['leiden'] = adata.obs['leiden'].values

# 5. 计算每个leiden簇的平均表达水平
print("Calculating mean expression per leiden cluster...")
mean_expression = expression_df.groupby('leiden')[existing_genes].mean()

# 6. 添加top5列
def get_top5_genes(row):
    """获取表达量最高的5个基因"""
    top5 = row.sort_values(ascending=False).head(5)
    return ', '.join(top5.index.tolist())

print("Adding top5 genes column...")
mean_expression['top5'] = mean_expression.apply(get_top5_genes, axis=1)

# 7. 创建基因分类行（用于添加到数据框顶部）
gene_classification_row = pd.DataFrame(
    [['Gene_Type'] + [gene_to_subtype[gene] for gene in existing_genes] + ['']],
    columns=['leiden'] + existing_genes + ['top5']
)

# 8. 合并分类行和平均表达数据
mean_expression_reset = mean_expression.reset_index()
final_df = pd.concat([gene_classification_row, mean_expression_reset], ignore_index=True)
# 9. 设置列顺序
final_df = final_df[['leiden'] + existing_genes + ['top5']]
# 10. 创建 leiden_ct_df (从adata中提取)
leiden_ct_df = adata.obs[['leiden', 'Celltype']].drop_duplicates().sort_values('leiden')
print(f"\nLeiden-Celltype mapping:\n{leiden_ct_df}")
# 11. 合并 final_df 和 leiden_ct_df
# 注意：需要保留分类行（leiden = 'Gene_Type'），然后为其他行添加Celltype信息
# 分离分类行和数据行
classification_row = final_df[final_df['leiden'] == 'Gene_Type'].copy()
data_rows = final_df[final_df['leiden'] != 'Gene_Type'].copy()
# 将leiden列转换为数值类型以便合并
# data_rows['leiden'] = pd.to_numeric(data_rows['leiden'])  # 不用，leiden_ct_df 中的也是 str
# 合并数据行与Celltype信息
data_rows_merged = data_rows.merge(leiden_ct_df, on='leiden', how='left')
# 检查是否有缺失的Celltype
missing_ct = data_rows_merged[data_rows_merged['Celltype'].isna()]
if len(missing_ct) > 0:
    print(f"\nWarning: {len(missing_ct)} leiden clusters missing Celltype information:")
    print(missing_ct[['leiden']])

# 重新排列列顺序：将Celltype放在leiden之后，top5之前
cols = ['leiden', 'Celltype'] + existing_genes + ['top5']
data_rows_merged = data_rows_merged[cols]
# 将分类行的Celltype列填充为''
classification_row['Celltype'] = ''

# 重新合并分类行和数据行，保持分类行在第一行
final_df_new = pd.concat([classification_row, data_rows_merged], ignore_index=True)
#### 再把 top5 DEGs的信息加上
Fib_leiden_degs = pd.read_csv('/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_100degs.tsv',sep='\t')
top5_degs_per_group = Fib_leiden_degs.groupby('group').head(5)
top5_degs_agg = top5_degs_per_group.groupby('group')['names'].agg(lambda x: ', '.join(x)).reset_index()
top5_degs_agg.columns = ['leiden', 'top5DEG']
classification_row_deg = pd.DataFrame({
    'leiden': ['Gene_Type'],
    'top5DEG': ['']
})
# 合并分类行和数据行
leiden_deg = pd.concat([classification_row_deg, top5_degs_agg], ignore_index=True)
final_df_new['leiden'] = final_df_new['leiden'].astype(str)
leiden_deg['leiden'] = leiden_deg['leiden'].astype(str)
final_df_new_with_degs = final_df_new.merge(leiden_deg, on='leiden', how='left')
cols = final_df_new_with_degs.columns.tolist()
# 找到 top5 列的位置
top5_idx = cols.index('leiden')
# 把 Celltype, top5, top5DEG 这几列都加入到 leiden列后面,便于比较
cols.insert(top5_idx + 1, cols.pop(cols.index('Celltype')))
cols.insert(top5_idx + 2, cols.pop(cols.index('top5')))
cols.insert(top5_idx + 3, cols.pop(cols.index('top5DEG')))
final_df_new_with_degs = final_df_new_with_degs[cols]
final_df_new_with_degs.to_csv('external_inputs/Fibro_marker_all_exp.csv',sep=',',index=False)  # 然后自己去 WPS表格里进行高亮着色分块，隐藏滑动等操作
print("\n✅ Analysis complete!")



In [ ]:
###### 画 Fibroblast marker list的图，一个gene 一个 UMAP
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

union_list = [gene for gene in Fib_marker_all if gene in adata.var_names]   ### 筛选 adata中有的 gene
n_cols =    9
n_rows =    6
figsize = (23, 12)  # 总画布大小

# 创建图形
fig = plt.figure(figsize=figsize)

# 使用 GridSpec 创建子图网格
gs = GridSpec(n_rows, n_cols, figure=fig, wspace=0.3, hspace=0.4)

# 对每个基因绘制 UMAP
for idx, gene in enumerate(union_list):
    row = idx // n_cols
    col = idx % n_cols
    ax = fig.add_subplot(gs[row, col])
    # 绘制 UMAP
    sc.pl.umap(adata, color=gene, ax=ax, show=False, size=2,
               title=gene, frameon=True,
               legend_loc='none')  # 隐藏颜色条，减少拥挤
    
    # 自定义标题和标签
    ax.set_title(gene, fontsize=8, fontweight='bold', pad=2)
    ax.set_xlabel('', fontsize=6)
    ax.set_ylabel('', fontsize=6)
    ax.tick_params(labelsize=5)
    
    # 移除坐标轴标签（可选，使图更简洁）
    ax.set_xlabel('')
    ax.set_ylabel('')

# 隐藏多余的子图
for idx in range(len(union_list), n_rows * n_cols):
    row = idx // n_cols
    col = idx % n_cols
    ax = fig.add_subplot(gs[row, col])
    ax.axis('off')

# 添加总标题
fig.suptitle('Endothelial marker gene umap', 
             fontsize=14, y=0.98)

# 调整布局
plt.tight_layout()
plt.subplots_adjust(top=0.96, bottom=0.02, left=0.02, right=0.98)
fig_dir='external_inputs/umap'
fig_name = 'Fibro_markers.png'
plt.savefig(fig_dir+fig_name, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()


In [ ]:
print(base_output_dir)
col ='leiden'
sc.tl.rank_genes_groups(
    adata, 
    groupby=col, 
    method='wilcoxon', 
    n_genes=None,          # 返回所有基因
    key_added=f'DEG_{col}'  # 避免覆盖其他分组的结果
)
# 获取该列的所有分组类别
groups = adata.obs[col].cat.categories.tolist()
# 提取所有分组的差异表达结果（合并为一个DataFrame）
df_all = sc.get.rank_genes_groups_df(
    adata, 
    group=groups, 
    key=f'DEG_{col}'
)
# 保存结果到tsv文件
output_file = f"{base_output_dir}Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_{col}_allDEGs.tsv"
df_all.to_csv(output_file, sep='\t', index=False)
print(f"已保存 {col} 的DEG结果到：{output_file}")
del adata
gc.collect()
# 可选：如果想同时保存每个分组单独的表格，可以循环groups分别提取
# for g in groups:
#     df_g = sc.get.rank_genes_groups_df(adata, group=g, key=f'rank_genes_{col}')
#     df_g.to_csv(f"{base_output_dir}{col}_{g}_allgenes.tsv", sep='\t', index=False)



In [ ]:
import scanpy as sc
import pandas as pd

# 假设 adata 已经加载，如果还没有，先读取
adata_file_path =['/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyFib.h5ad','/data/project/chenfeiwang/ytwang/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyEndo.h5ad']
lineage_index = 0
if lineage_index ==0:
    lineage = 'Fibro'
else:
    lineage ='Endo'

    
adata = sc.read_h5ad(adata_file_path[lineage_index])
group_columns = ['leiden', 'Celltype', 'Celltype_general']
base_output_dir = 'external_inputs/figure'+lineage+'/' +'DEGs/'

for col in group_columns:
    if col not in adata.obs.columns:
        print(f"警告：'{col}' 不在 adata.obs 中，跳过")
        continue
    print(f"正在处理分组列：{col}")
    # 执行差异表达分析，返回所有基因（n_genes=None）
    sc.tl.rank_genes_groups(
        adata, 
        groupby=col, 
        method='wilcoxon', 
        n_genes=None,          # 返回所有基因
        key_added=f'DEG_{col}'  # 避免覆盖其他分组的结果
    )
    # 获取该列的所有分组类别
    groups = adata.obs[col].cat.categories.tolist()
    # 提取所有分组的差异表达结果（合并为一个DataFrame）
    df_all = sc.get.rank_genes_groups_df(
        adata, 
        group=groups, 
        key=f'DEG_{col}'
    )
    # 保存结果到tsv文件
    output_file = f"{base_output_dir}Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_{col}_allDEGs.tsv"
    df_all.to_csv(output_file, sep='\t', index=False)
    print(f"已保存 {col} 的DEG结果到：{output_file}")
    gc.collect()
    # 可选：如果想同时保存每个分组单独的表格，可以循环groups分别提取
    # for g in groups:
    #     df_g = sc.get.rank_genes_groups_df(adata, group=g, key=f'rank_genes_{col}')
    #     df_g.to_csv(f"{base_output_dir}{col}_{g}_allgenes.tsv", sep='\t', index=False)

print("所有分组列的DEG分析完成！")



In [ ]:
######## 得到每个 leiden 中 top 100 的 DEGs, 并保存 adata
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon',n_genes=100)
adata.write('/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn.h5ad')
groups = adata.obs['leiden'].cat.categories
df = sc.get.rank_genes_groups_df(adata, group=groups)
df.to_csv('/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_100degs.tsv',sep='\t',index=False)
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="leiden", standard_scale="var", n_genes=5
)
fig_dir='external_inputs/annotation'
fig_name = 'Fibro_top5DEGs.png'
plt.savefig(fig_dir+fig_name, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
## 重新再算一下
Fib_marker_deg = ['DCN', 'COL1A2', 'LUM', 'MGP', 'COL1A1', 'COL3A1', 'CALD1', 'IGFBP7', 'COL6A2', 'SPARC']
Endo_marker_deg =['VWF', 'PECAM1', 'TM4SF1', 'RAMP2', 'GNG11', 'CDH5', 'TEK', 'KDR']  
gene_marker_list = [gene for gene in Fib_marker_deg if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Fib_deg')
gene_marker_list = [gene for gene in Endo_marker_deg if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Endo_deg')


signature_cols = ['MS_Fib_deg', 'MS_Endo_deg', 'Fib_deg', 'Endo_deg','T', 'B', 'Fibro', 'Endo', 'MoMac', 'Epi', 
                  'Myfibro', 'eFibro', 'iFibro', 'qFibro', 'Myfibro1', 'eFibro1',
       'iFibro1', 'qFibro1', 'ECM', 'Immune', 'AP']
score_df = adata.obs.groupby('leiden')[signature_cols].mean().reset_index()
score_df.to_csv('external_inputs/score_by_leiden.csv',sep=',',index=False)
score_df = pd.read_csv('external_inputs/score_by_leiden.csv',sep=',')
# score_df


In [ ]:

import pandas as pd
import numpy as np
import openpyxl 

def highlight_max_rowwise(row):
    max_val = row[signature_cols].max()
    return ['color: red; font-weight: bold' if v == max_val and pd.notna(v) else ''  for v in row[signature_cols]]

    
# ['MS_Fib_deg', 'MS_Endo_deg', 'Fib_deg', 'Endo_deg','T', 'B', 'Fibro', 'Endo', 'MoMac', 'Epi', 
#                   'Myfibro', 'eFibro', 'iFibro', 'qFibro', 'Myfibro1', 'eFibro1','iFibro1', 'qFibro1', 'ECM', 'Immune', 'AP']     # 这个是 score_df当时选的列

score_df = pd.read_csv('external_inputs/score_by_leiden.csv',sep=',')
signature_cols = [ 'Fib_deg', 'Endo_deg','T', 'B', 'Fibro', 'Endo', 'MoMac', 'Epi', 
                  'Myfibro', 'eFibro', 'iFibro', 'qFibro', 'Myfibro1', 'eFibro1','iFibro1', 'qFibro1', 'ECM', 'Immune', 'AP']
styled_score_df = score_df[signature_cols].style.apply(highlight_max_rowwise, axis=1)   ### 截取某些行的话就用  score_df[signature_cols].tail(25) 或者 score_df[signature_cols].head(30)

# 4. 设置更美观的显示格式（可选：设置数值精度）
styled_score_df = styled_score_df.format({col: '{:.4f}' for col in signature_cols})
styled_score_df



In [ ]:
# sorted(dir())

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

def calculate_metrics(y_true, y_pred):
    """计算各种评估指标"""
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mad = np.median(np.abs(y_true - y_pred))
    
    try:
        pcc, _ = pearsonr(y_true, y_pred)
    except:
        pcc = np.nan
    
    return {
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse,
        "MAD": mad,
        "PCC": pcc
    }

def cross_validation_evaluation(X, y, model, n_splits=5, random_state=42):
    """
    执行交叉验证评估
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    cv_results = {
        'train_r2': [],
        'valid_r2': [],
        'train_mae': [],
        'valid_mae': [],
        'train_pcc': [],
        'valid_pcc': []
    }
    
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X)):
        try:
            X_train_fold = X[train_idx]
            X_val_fold = X[val_idx]
            y_train_fold = y[train_idx]
            y_val_fold = y[val_idx]
            
            # 克隆模型参数
            model_params = model.get_params()
            model_fold = ElasticNet(**model_params)
            
            model_fold.fit(X_train_fold, y_train_fold)
            
            train_pred = model_fold.predict(X_train_fold)
            val_pred = model_fold.predict(X_val_fold)
            
            train_metrics = calculate_metrics(y_train_fold, train_pred)
            val_metrics = calculate_metrics(y_val_fold, val_pred)
            
            cv_results['train_r2'].append(train_metrics['R2'])
            cv_results['valid_r2'].append(val_metrics['R2'])
            cv_results['train_mae'].append(train_metrics['MAE'])
            cv_results['valid_mae'].append(val_metrics['MAE'])
            cv_results['train_pcc'].append(train_metrics['PCC'])
            cv_results['valid_pcc'].append(val_metrics['PCC'])
            
        except Exception as e:
            print(f"    Fold {fold_idx+1} 失败: {e}")
            continue
    
    if len(cv_results['train_r2']) > 0:
        stats = {
            'train_r2_mean': np.mean(cv_results['train_r2']),
            'train_r2_std': np.std(cv_results['train_r2']),
            'valid_r2_mean': np.mean(cv_results['valid_r2']),
            'valid_r2_std': np.std(cv_results['valid_r2']),
            'train_mae_mean': np.mean(cv_results['train_mae']),
            'train_mae_std': np.std(cv_results['train_mae']),
            'valid_mae_mean': np.mean(cv_results['valid_mae']),
            'valid_mae_std': np.std(cv_results['valid_mae']),
            'train_pcc_mean': np.mean(cv_results['train_pcc']),
            'train_pcc_std': np.std(cv_results['train_pcc']),
            'valid_pcc_mean': np.mean(cv_results['valid_pcc']),
            'valid_pcc_std': np.std(cv_results['valid_pcc']),
            'n_successful_folds': len(cv_results['train_r2'])
        }
        return stats
    else:
        return None

def build_elasticnet_optimized(
    df_features, 
    target_col="Age",
    individual_col="sample",
    lineage="unknown",
    test_size=0.2,
    random_state=42,
    variance_threshold=1e-6,  # 降低阈值，保留更多特征
    k_expr=15000,  # 增加特征数量
    n_age_bins=8
):  
    print("=" * 80)
    print("🔧 ElasticNet 欠拟合优化 - 参数网格搜索")
    print("=" * 80)
    
    data = df_features.copy()
    data[target_col] = pd.to_numeric(
        data[target_col],
        errors="coerce"
    ) 
    data = data.dropna(
        subset=[target_col]
    ).reset_index(drop=True)
    y = data[target_col].to_numpy(dtype=float)
    
    if individual_col in data.columns:
        individual_ids = data[individual_col].astype(str).to_numpy()
    else:
        individual_ids = np.arange(len(data)).astype(str)
    
    feature_cols = [
        col for col in data.columns
        if col not in [target_col, individual_col]
    ]  
    expr_cols = [col for col in feature_cols]
    
    print(f"总样本数：{len(data)}")
    print(f"原始表达特征数：{len(expr_cols)}")
    
    # =========================
    # 1. 拆分数据
    # =========================
    indices = np.arange(len(data))  
    try:
        age_bins = pd.qcut(
            pd.Series(y),  
            q=n_age_bins,
            labels=False,
            duplicates="drop"
        )    
        if age_bins.value_counts().min() < 2:
            age_bins = None
    except ValueError:
        age_bins = None  
    
    train_idx, test_idx = train_test_split(
        indices,
        test_size=test_size, 
        random_state=random_state,
        stratify=age_bins
    )  
    
    y_train = y[train_idx] 
    y_test = y[test_idx] 
    id_train = individual_ids[train_idx]
    id_test = individual_ids[test_idx]
    
    # =========================
    # 2. 提取特征
    # =========================
    X_expr_train = data.loc[
        train_idx,
        expr_cols
    ].to_numpy(dtype=np.float32) 
    X_expr_test = data.loc[
        test_idx,
        expr_cols
    ].to_numpy(dtype=np.float32)
    
    # =========================
    # 3. 特征筛选 - 保留更多特征
    # =========================
    vt = VarianceThreshold(
        threshold=variance_threshold
    )
    X_expr_train_vt = vt.fit_transform(
        X_expr_train
    )
    X_expr_test_vt = vt.transform(
        X_expr_test
    ) 
    expr_cols_after_vt = np.asarray(expr_cols)[vt.get_support()]
    
    # 使用更大的k值
    actual_k = min(
        k_expr,
        X_expr_train_vt.shape[1]
    )
    skb = SelectKBest(
        score_func=f_regression,   
        k=actual_k
    )
    X_expr_train_selected = skb.fit_transform(
        X_expr_train_vt,
        y_train
    )
    X_expr_test_selected = skb.transform(
        X_expr_test_vt
    ) 
    selected_expr_cols = expr_cols_after_vt[
        skb.get_support()
    ]
    
    print(f"低方差筛选后特征数：{len(expr_cols_after_vt)}")
    print(f"最终保留特征数：{len(selected_expr_cols)}")
    
    # =========================
    # 4. 标准化
    # =========================
    scaler_expr = StandardScaler()
    X_train_scaled = scaler_expr.fit_transform(
        X_expr_train_selected
    )
    X_test_scaled = scaler_expr.transform(
        X_expr_test_selected
    )
    
    final_feature_names = list(selected_expr_cols)
    
    print(f"训练集矩阵：{X_train_scaled.shape}")
    print(f"测试集矩阵：{X_test_scaled.shape}")
    print("=" * 80)
    
    # =========================
    # 5. ElasticNet 参数网格设计
    # 针对欠拟合：降低正则化强度，增加模型复杂度
    # =========================
    
    # 5.1 精细的alpha参数 (从非常小到中等)
    alpha_values = [
        0.001, 0.002, 0.005, 0.01, 0.02, 0.03, 0.05, 
        0.08, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0
    ]
    
    # 5.2 l1_ratio参数 (控制L1/L2比例)
    l1_ratio_values = [
        0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 
        0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99
    ]
    
    # 5.3 构建参数网格 - 策略1: 固定l1_ratio, 变化alpha
    param_grid_alpha = []
    fixed_l1 = 0.5  # 平衡L1和L2
    for alpha in alpha_values:
        param_grid_alpha.append({
            "alpha": alpha,
            "l1_ratio": fixed_l1,
            "param_strategy": "alpha_variation"
        })
    
    # 5.4 构建参数网格 - 策略2: 固定alpha, 变化l1_ratio
    param_grid_l1 = []
    fixed_alpha = 0.01  # 非常小的正则化
    for l1_ratio in l1_ratio_values:
        param_grid_l1.append({
            "alpha": fixed_alpha,
            "l1_ratio": l1_ratio,
            "param_strategy": "l1_variation"
        })
    
    # 5.5 构建参数网格 - 策略3: 极低正则化组合
    param_grid_low = [
        {"alpha": 0.001, "l1_ratio": 0.01, "param_strategy": "very_low"},
        {"alpha": 0.001, "l1_ratio": 0.1, "param_strategy": "very_low"},
        {"alpha": 0.001, "l1_ratio": 0.5, "param_strategy": "very_low"},
        {"alpha": 0.002, "l1_ratio": 0.01, "param_strategy": "very_low"},
        {"alpha": 0.002, "l1_ratio": 0.1, "param_strategy": "very_low"},
        {"alpha": 0.005, "l1_ratio": 0.01, "param_strategy": "very_low"},
        {"alpha": 0.005, "l1_ratio": 0.1, "param_strategy": "very_low"},
        {"alpha": 0.005, "l1_ratio": 0.5, "param_strategy": "very_low"},
        {"alpha": 0.01, "l1_ratio": 0.01, "param_strategy": "very_low"},
        {"alpha": 0.01, "l1_ratio": 0.1, "param_strategy": "very_low"},
        {"alpha": 0.01, "l1_ratio": 0.5, "param_strategy": "very_low"},
        {"alpha": 0.02, "l1_ratio": 0.01, "param_strategy": "very_low"},
        {"alpha": 0.02, "l1_ratio": 0.1, "param_strategy": "very_low"},
    ]
    
    # 5.6 策略4: 在最佳参数附近细化搜索
    param_grid_fine = [
        {"alpha": 0.005, "l1_ratio": 0.2, "param_strategy": "fine_tune"},
        {"alpha": 0.005, "l1_ratio": 0.3, "param_strategy": "fine_tune"},
        {"alpha": 0.005, "l1_ratio": 0.4, "param_strategy": "fine_tune"},
        {"alpha": 0.01, "l1_ratio": 0.2, "param_strategy": "fine_tune"},
        {"alpha": 0.01, "l1_ratio": 0.3, "param_strategy": "fine_tune"},
        {"alpha": 0.01, "l1_ratio": 0.4, "param_strategy": "fine_tune"},
        {"alpha": 0.02, "l1_ratio": 0.2, "param_strategy": "fine_tune"},
        {"alpha": 0.02, "l1_ratio": 0.3, "param_strategy": "fine_tune"},
        {"alpha": 0.02, "l1_ratio": 0.4, "param_strategy": "fine_tune"},
        {"alpha": 0.03, "l1_ratio": 0.2, "param_strategy": "fine_tune"},
        {"alpha": 0.03, "l1_ratio": 0.3, "param_strategy": "fine_tune"},
        {"alpha": 0.03, "l1_ratio": 0.4, "param_strategy": "fine_tune"},
    ]
    
    # 合并所有参数网格
    all_params = param_grid_alpha + param_grid_l1 + param_grid_low + param_grid_fine
    
    # 去除重复的参数组合
    unique_params = []
    seen = set()
    for params in all_params:
        key = (params['alpha'], params['l1_ratio'])
        if key not in seen:
            seen.add(key)
            unique_params.append(params)
    
    print(f"总共测试 {len(unique_params)} 个ElasticNet参数组合")
    print("=" * 80)
    
    # =========================
    # 6. 训练和评估所有参数组合
    # =========================
    results = []
    fitted_models = {}
    best_cv_r2 = -np.inf
    best_params = None
    
    for idx, params in enumerate(unique_params):
        alpha = params['alpha']
        l1_ratio = params['l1_ratio']
        strategy = params['param_strategy']
        
        model_name = f"EN_a{alpha:.4f}_l1{l1_ratio:.3f}"
        
        print(f"\n[{idx+1}/{len(unique_params)}] 训练: {model_name}")
        print(f"  alpha={alpha:.4f}, l1_ratio={l1_ratio:.3f}")
        print(f"  策略: {strategy}")
        
        start_time = time.time()
        
        try:
            # 创建ElasticNet模型 - 降低正则化
            model = ElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                max_iter=1000000,  # 增加迭代次数
                random_state=random_state,
                tol=1e-6,  # 更严格的收敛条件
                warm_start=False
            )
            
            # 在完整训练集上训练
            model.fit(X_train_scaled, y_train)
            
            # 预测
            train_pred = model.predict(X_train_scaled)
            test_pred = model.predict(X_test_scaled)
            
            # 计算指标
            train_metrics = calculate_metrics(y_train, train_pred)
            test_metrics = calculate_metrics(y_test, test_pred)
            
            # 5折交叉验证
            print(f"  进行5折交叉验证...")
            cv_stats = cross_validation_evaluation(
                X_train_scaled, 
                y_train, 
                model, 
                n_splits=5,
                random_state=random_state
            )
            
            runtime_min = (time.time() - start_time) / 60
            
            # 保存结果
            result = {
                "Lineage": lineage,
                "Model": model_name,
                "Alpha": alpha,
                "L1_Ratio": l1_ratio,
                "Param_Strategy": strategy,
                "Feature_Count": len(selected_expr_cols),
                
                # 训练集结果
                "Train_R2": train_metrics['R2'],
                "Train_MAE": train_metrics['MAE'],
                "Train_RMSE": train_metrics['RMSE'],
                "Train_PCC": train_metrics['PCC'],
                
                # 测试集结果
                "Test_R2": test_metrics['R2'],
                "Test_MAE": test_metrics['MAE'],
                "Test_RMSE": test_metrics['RMSE'],
                "Test_PCC": test_metrics['PCC'],
                
                "Runtime_min": runtime_min
            }
            
            # 添加交叉验证结果
            if cv_stats:
                result.update({
                    "CV_Train_R2_Mean": cv_stats['train_r2_mean'],
                    "CV_Train_R2_Std": cv_stats['train_r2_std'],
                    "CV_Valid_R2_Mean": cv_stats['valid_r2_mean'],
                    "CV_Valid_R2_Std": cv_stats['valid_r2_std'],
                    "CV_Valid_MAE_Mean": cv_stats['valid_mae_mean'],
                    "CV_Valid_MAE_Std": cv_stats['valid_mae_std'],
                    "CV_Train_PCC_Mean": cv_stats['train_pcc_mean'],
                    "CV_Valid_PCC_Mean": cv_stats['valid_pcc_mean'],
                    "Train_Valid_Gap": cv_stats['train_r2_mean'] - cv_stats['valid_r2_mean']
                })
            else:
                result.update({
                    "CV_Train_R2_Mean": np.nan,
                    "CV_Train_R2_Std": np.nan,
                    "CV_Valid_R2_Mean": np.nan,
                    "CV_Valid_R2_Std": np.nan,
                    "CV_Valid_MAE_Mean": np.nan,
                    "CV_Valid_MAE_Std": np.nan,
                    "CV_Train_PCC_Mean": np.nan,
                    "CV_Valid_PCC_Mean": np.nan,
                    "Train_Valid_Gap": np.nan
                })
            
            results.append(result)
            fitted_models[model_name] = model
            
            print(f"  ✅ Train R²: {train_metrics['R2']:.4f}")
            print(f"  ✅ Test R²: {test_metrics['R2']:.4f}")
            if cv_stats:
                print(f"  ✅ CV Valid R²: {cv_stats['valid_r2_mean']:.4f} ± {cv_stats['valid_r2_std']:.4f}")
                print(f"  ✅ Train-Valid Gap: {result['Train_Valid_Gap']:.4f}")
            print(f"  ✅ Runtime: {runtime_min:.2f} min")
            
            # 更新最佳模型
            if cv_stats and cv_stats['valid_r2_mean'] > best_cv_r2:
                best_cv_r2 = cv_stats['valid_r2_mean']
                best_params = {'alpha': alpha, 'l1_ratio': l1_ratio}
            
        except Exception as e:
            print(f"  ❌ 训练失败: {e}")
            continue
    
    # =========================
    # 7. 结果分析
    # =========================
    results_df = pd.DataFrame(results)
    
    if len(results_df) == 0:
        print("\n❌ 所有模型训练失败！")
        return None, None, None, None
    
    # 按训练R²排序（解决欠拟合）
    results_df = results_df.sort_values('Train_R2', ascending=False).reset_index(drop=True)
    
    print("\n" + "=" * 80)
    print("📊 所有ElasticNet结果汇总 (按训练R²排序)")
    print("=" * 80)
    
    display_cols = ['Model', 'Alpha', 'L1_Ratio', 'Train_R2', 'Test_R2', 
                    'CV_Valid_R2_Mean', 'Train_Valid_Gap', 'Test_MAE']
    available_cols = [col for col in display_cols if col in results_df.columns]
    print(results_df[available_cols].head(20).to_string(index=False))
    
    # =========================
    # 8. 按策略分组分析
    # =========================
    print("\n" + "=" * 80)
    print("📊 按参数策略分组统计")
    print("=" * 80)
    
    for strategy in results_df['Param_Strategy'].unique():
        strategy_results = results_df[results_df['Param_Strategy'] == strategy]
        best_in_strategy = strategy_results.iloc[0]
        print(f"\n策略: {strategy}")
        print(f"  测试组合数: {len(strategy_results)}")
        print(f"  最佳模型: {best_in_strategy['Model']}")
        print(f"  Train R²: {best_in_strategy['Train_R2']:.4f}")
        print(f"  Test R²: {best_in_strategy['Test_R2']:.4f}")
        if 'CV_Valid_R2_Mean' in best_in_strategy and not pd.isna(best_in_strategy['CV_Valid_R2_Mean']):
            print(f"  CV Valid R²: {best_in_strategy['CV_Valid_R2_Mean']:.4f}")
    
    # =========================
    # 9. 最佳模型分析
    # =========================
    best_model = results_df.iloc[0]
    
    print("\n" + "=" * 80)
    print("🏆 最佳ElasticNet模型 (解决欠拟合)")
    print("=" * 80)
    print(f"模型名称: {best_model['Model']}")
    print(f"Alpha: {best_model['Alpha']:.4f}")
    print(f"L1_Ratio: {best_model['L1_Ratio']:.3f}")
    print(f"参数策略: {best_model['Param_Strategy']}")
    
    print(f"\n📈 训练集性能:")
    print(f"  R²: {best_model['Train_R2']:.4f}")
    print(f"  MAE: {best_model['Train_MAE']:.3f}")
    print(f"  RMSE: {best_model['Train_RMSE']:.3f}")
    print(f"  PCC: {best_model['Train_PCC']:.4f}")
    
    print(f"\n📈 测试集性能:")
    print(f"  R²: {best_model['Test_R2']:.4f}")
    print(f"  MAE: {best_model['Test_MAE']:.3f}")
    print(f"  RMSE: {best_model['Test_RMSE']:.3f}")
    print(f"  PCC: {best_model['Test_PCC']:.4f}")
    
    if 'CV_Valid_R2_Mean' in best_model and not pd.isna(best_model['CV_Valid_R2_Mean']):
        print(f"\n📈 交叉验证性能:")
        print(f"  Valid R²: {best_model['CV_Valid_R2_Mean']:.4f} ± {best_model['CV_Valid_R2_Std']:.4f}")
        print(f"  Train-Valid Gap: {best_model['Train_Valid_Gap']:.4f}")
    
    # 判断是否解决了欠拟合
    print(f"\n🔍 欠拟合状态评估:")
    if best_model['Train_R2'] < 0.5:
        print(f"  ⚠️  训练R² ({best_model['Train_R2']:.3f}) 仍然较低")
        print(f"  建议: 进一步降低alpha值 (尝试 < {best_model['Alpha']:.4f})")
        print(f"  或增加特征数量 (当前 {best_model['Feature_Count']})")
    elif best_model['Train_R2'] < 0.7:
        print(f"  📊 训练R² ({best_model['Train_R2']:.3f}) 中等，仍有提升空间")
        print(f"  建议: 尝试更小的alpha值或增加更多特征")
    else:
        print(f"  ✅ 训练R² ({best_model['Train_R2']:.3f}) 表现良好")
        print(f"  欠拟合问题已得到改善!")
    
    # =========================
    # 10. 参数敏感性分析
    # =========================
    print("\n" + "=" * 80)
    print("📊 参数敏感性分析")
    print("=" * 80)
    
    # Alpha变化的影响 (固定l1_ratio)
    alpha_fixed = results_df[results_df['Param_Strategy'] == 'alpha_variation']
    if len(alpha_fixed) > 0:
        print("\nAlpha变化 (固定l1_ratio=0.5):")
        for _, row in alpha_fixed.iterrows():
            print(f"  alpha={row['Alpha']:.4f}: Train R²={row['Train_R2']:.4f}, Test R²={row['Test_R2']:.4f}")
    
    # L1_Ratio变化的影响 (固定alpha)
    l1_fixed = results_df[results_df['Param_Strategy'] == 'l1_variation']
    if len(l1_fixed) > 0:
        print("\nL1_Ratio变化 (固定alpha=0.01):")
        for _, row in l1_fixed.iterrows():
            print(f"  l1_ratio={row['L1_Ratio']:.3f}: Train R²={row['Train_R2']:.4f}, Test R²={row['Test_R2']:.4f}")
    
    # =========================
    # 11. 保存结果
    # =========================
    results_df.to_csv('elasticnet_欠拟合优化结果.csv', index=False)
    print(f"\n✅ 结果已保存到 'elasticnet_欠拟合优化结果.csv'")
    
    # 保存最佳模型的预测
    best_model_name = best_model['Model']
    best_model_obj = fitted_models.get(best_model_name)
    if best_model_obj:
        # 在测试集上保存预测结果
        test_pred = best_model_obj.predict(X_test_scaled)
        predictions_df = pd.DataFrame({
            'sample': id_test,
            'Age': y_test,
            'Pred_Age': test_pred
        })
        predictions_df.to_csv('best_elasticnet_predictions.csv', index=False)
        print("✅ 最佳模型预测结果已保存")
    
    preprocessing_objects = {
        "expr_cols": list(expr_cols),
        "variance_selector": vt,
        "select_k_best": skb,
        "selected_expr_cols": list(selected_expr_cols),
        "scaler_expr": scaler_expr,
        "final_feature_names": final_feature_names,
        "train_indices": train_idx,
        "test_indices": test_idx
    }
    
    split_data = {
        "X_train": X_train_scaled,
        "X_test": X_test_scaled,
        "y_train": y_train,
        "y_test": y_test,
        "id_train": id_train,
        "id_test": id_test
    }
    
    return (
        results_df,
        fitted_models,
        preprocessing_objects,
        split_data
    )

# =========================
# 运行主程序
# =========================
print("开始ElasticNet欠拟合优化...")
print("=" * 80)

(
    results_df,
    fitted_models,
    preprocessing_objects,
    split_data
) = build_elasticnet_optimized(
    df_features=df_features,
    lineage=lineage,
    target_col="Age",
    individual_col="sample",
    test_size=0.2,
    random_state=42,
    variance_threshold=1e-6,  # 降低阈值，保留更多特征
    k_expr=15000,  # 增加特征数量
    n_age_bins=8
)

print("\n" + "=" * 80)
print("✅ ElasticNet欠拟合优化完成!")
print("=" * 80)


In [ ]:
signature_cols = ['Fib_deg', 'Endo_deg','T', 'B', 'Fibro', 'Endo', 'MoMac', 'Epi', 
                  'Myfibro', 'eFibro', 'iFibro', 'qFibro', 'Myfibro1', 'eFibro1','iFibro1', 'qFibro1', 'ECM', 'Immune', 'AP']
styled_score_df = score_df[signature_cols].tail(30).style.apply(highlight_max_rowwise, axis=1)   ### 截取某些行的话就用  score_df[signature_cols].tail(25) 或者 score_df[signature_cols].head(30)
styled_score_df.format({col: '{:.4f}' for col in signature_cols})


In [ ]:
######## 算了 rank DEG之后，展示某个subtype中不同Leiden的top5 DEGs
metadata = adata.obs
int_list = sorted([int(x) for x in metadata[metadata['Celltype_general']=='eFibro']['leiden'].unique()])
eFibro_deg = df[df['group'].isin(int_list)].groupby('group').head(5)   # qFibro, iFibro, Myfibro 同理



In [ ]:
########## 然后注释，分别对每个 leiden 进行注释
######## 对 adata 中的不同 leiden进行 subtype 注释, 需要先画下每个leiden 中6+8 个signature score的图
######### 对Fibro进行重新注释，这里是 4个 subtype的

cluster_to_celltype = {  
  '0': 'eFibro_COL1A2_pan', 
  '1': 'eFibro_COL1A1_pan', 
  '2': 'qFibro_APOD_breast',   # signature 是 qFibro最高，但eFibro的marker表达很高; leiden 3也是同理
  '3': 'qFibro_CFD_pan', 
  '4': 'eFibro_DCN_pan', 
  '5': 'Myfibro_TAGLN_pan',  # IGFBP7是ECM，也就是eFibro经典marker，但signature score注释为 Myfibro,check下; Fibro marker中top5为 IGFBP7、CALD1、SPARC、TAGLN、MGP;DEG中top5为['IGFBP7','RGS5','CALD1','NDUFA4L2','NOTCH3']
  '6': 'Epi_KRT18_breast',   # 
  '7': 'Myfibro_TAGLN_pan',
  '8': 'Myfibro_TUBB2B_pan',   #  top5 DEG为['TUBB2B', 'TUBA1A', 'CKB', 'MAP1B', 'MARCKSL1'], 可以是Myfibro的marker，TUBB2B、TUBA1A、MAP1B、MARCKSL1 是肌成纤维活化、迁移的高度富集基因
  '9': 'eFibro_HNRNPA1_pan',  # HNRNPA1 表达排序 ——eFibro ≥ Myfibro > iFibro >> qFibro,HNRNPA1 在所有组织的成纤维细胞中均高表达,属 “管家型” 高丰度 RNA 结合蛋白
  '10': 'eFibro_COL1A1_pan',  
  '11': 'eFibro_COL1A1_lung',  
  '12': 'eFibro_COL3A1_pan',   # ***  RPL17 表达排序：MyFibro ≥ eFibro > iFibro > qFibro; sc.pl.violin(adata, ['RPL17'], groupby='Celltype_general') ;sc.pl.dotplot(adata, ['RPL17'], groupby='Celltype_general') 
  '13': 'Endo_PECAM1_pan',   
  '14': 'eFibro_COL3A1_pan', 
  '15': 'eFibro_DCN_heart',  # 改:eFibro,    CLU也蛮高的, top5 DEG:['CLU', 'MGP', 'ITM2B', 'TIMP3', 'LTBP4'] ; Fib marker top5是 MGP, DCN, CLU, IGFBP7, COL1A2
  '16': 'Myfibro_KRT8_pan',     # 改:Myfibro(Epi差不太多)
  '17': 'iFibro_CD74_bone marrow', # 画 dotplot加上; top5 DEG: ['GMFG', 'ARHGDIB', 'AIF1', 'RPS3A', 'RPS3'], top5 Fibmarker是 CD74, ARHGDIB, HLA-DRA, IGFBP7, CFD
  '18': 'Epi_EPCAM_pan', # DEG: ['MT-CO1', 'MT-CO2', 'MT-CO3', 'MT-ND4', 'MT-CYB']这些基因在Fibro中有表达, MT-CO1 不是细胞亚型特异性标记基因，不能用来注释成纤维 / 上皮
  '19': 'Epi_KRT19_pan', 
  '20': 'iFibro_CD74_bone marrow',  # top5 DEG:['ARHGDIB', 'GMFG', 'EEF1G', 'RPL28', 'RPL17'], top5 Fibmarker是 CD74, ARHGDIB, HLA-DRA, IGFBP7, CFD
  '21': 'eFibro_DCN_pan', # DEG和 top5 Fib marker中都有这个基因，但DCN 为eFibro的marker；SS中Myfibro只比eFibro高一点, 改成 eFibro
  '22': 'eFibro_COL1A2_pan',   # 改:eFibro,     但 CD74应该在 iFibro 表达比较高，它和20,23都在一起，看下signature score，是不是注释成iFibro？iFibro的signature score也不低
  '23': 'iFibro_CD74_pan',  # top5 DEG: ['CD74', 'TYROBP', 'LYZ', 'SRGN', 'HLA-DRA'], top5 Fibmarker是 CD74, HLA-DRA, ARHGDIB, CFD, IGFBP7
  '24': 'eFibro_COL1A1_pan',  # FTH1 表达排序：eFibro ≥ iFibro > Myfibro > qFibro;
  '25': 'qFibro_CLU_pan', # Fibmarker中top5为:DCN, CLU, MGP, APOD, CFD; DCN为eFibro,其他几个都是qFibro的; SS注释是 qFibro
  '26': 'Myfibro_TPM1_pan',   # TPM1: MyFibro≫eFibro>iFibro≈qFibro, TPM1 与 TAGLN、CALD1、ACTG2 高度共表达，是肌成纤维细胞经典标志。
  '27': 'Myfibro_QKI_brain',  # 改:Myfibro
  '28': 'Epi_WFDC2_pan',  #     WFDC2在 Epithelial中表达极高
  '29': 'Myfibro_CALD1_kidney', # CALD1 + TAGLN + ACTA2 是 Myfibro三大经典核心 marker, CALD1在Myfibro中表达最高
  '30': 'eFibro_IGFBP7_pan',  # top10 DEG:['MT-CO1','MT-CO3','ATP1A1','MT-CYB','MT-CO2','ST3GAL4','ATP6V1G1','PEBP1','GMPR','MT-ND1'],Fibmarker top5:IGFBP7,SPARC,CALD1,ITGB1,COL1A2
 # MT系列呼吸链 + 能量代谢基因 在 eFibro 显著特异性高表达, 所以从 Mono/Macro 改成 eFibro
  '31': 'iFibro_ARHGDIB_pan', # top5 DEG:['PTPRC', 'B2M', 'HLA-B', 'HLA-C', 'SRGN'], Fibmarker top5:CCL5, CCL4, ARHGDIB, CD74, ITGB1
  '32': 'qFibro_CLU_pan',   ## 
  '33': 'qFibro_CLU_bone', # DEG第一个就是CLU,signature score中qFibro比eFibro高一点，改成 qFibro
  '34': 'Myfibro_MYH11_uterus',  # 后续画dotplot考虑将 MYH11 加进去，DEG & FibMarker top5 都有这个基因
  '35': 'Epi_KRT18_pan',  
  '36': 'eFibro_COL1A2_pan',  
  '37': 'Endo_EGFL7_pan',  
  '38': 'eFibro_COL1A2_pan',  
  '39': 'Myfibro_CTSA_skin',  # ***    改:Myfibro
  '40': 'Myfibro_TAGLN_pan',  
  '41': 'Myfibro_CALD1_pan',  # 改:Myfibro
  '42': 'Epi_KRT18_pan', 
  '43': 'Myfibro_TAGLN_pan',
  '44': 'Myfibro_TPM1_pan',  # 改:Myfibro
  '45': 'Myfibro_CALD1_pan', # 改:Myfibro
  '46': 'eFibro_SPARC_pan', # 改:eFibro,      top10 DEG:['GPM6B','S100B','CRYAB','PRNP','TUBA1A','PMP22','METRN','CD9','CLU','CNP'], top5 fibmarker:CLU, SPARC, IGFBP7, COL1A2, ITGB1; 注释成 qFibro
  '47': 'Epi_EPCAM_pan',  
  '48': 'Myfibro_CALD1_pan',  
  '49': 'eFibro_COL3A1_lung', #  RPL4: MyoFibro > eFibro > iFibro > qFibro，是 ** 增殖 / 活化（尤其 MyoFibro
  '50': 'Myfibro_CALD1_pan', # DEG:['DCBLD2', 'NME2', 'CALU', 'ITGB1', 'RNASEK']显示是eFibro;  Fibmarker top5是 COL1A1, CALD1, TAGLN, ITGB1, COL1A2; SS中Myfibro比eFibro高一点
  '51': 'Myfibro_CALD1_pan',   # 改:Myfibro
  '52': 'Myfibro_CDKN2A_brain', # 改:Myfibro(其实和eFibro差不太多),      CDKN2A 在 衰老活化成纤维细胞显著高表达, top5 DEG:['CDKN2A', 'UQCRH', 'ALDOA', 'IFI6', 'IGFBP2'],是一套典型的「活化 / 应激 / 修复态 Fibroblast」marker组,主体归属 eFibro
  '53': 'Myfibro_TUBB2B_pan',  # 改:Myfibro, 
  '54': 'eFibro_DCN_pan',
  '55': 'Myfibro_QKI_eye', #  改:Myfibro
  '56': 'eFibro_COL1A1_pan', 
  '57': 'Myfibro_QKI_brain',  # 改:Myfibro(其实和qFibro差不太多)      marker需要重新选下,  Fibmarker top5是:CALD1, APOD, CLU, MYLK, ITGB1; top5 DEG是 ['PIP4K2A', 'QKI', 'SLC44A1', 'MBP', 'PCDH9']
  '58': 'eFibro_DCN_uterus', # 改:eFibro      top5 DEG是['RAMP1', 'TGM2', 'RBP1', 'IGFBP4', 'LGALS3'], FibMarker top5是: DCN, IGFBP7, CFD, CALD1, APOD; TGM2、IGFBP4 为经典促纤维化与肌成纤维细胞分化标志物,改成 Myfibro
  '59': 'Myfibro_CALD1_pan', # 
  '60': 'iFibro_CD74_skin', # top5 DEG是['S100B', 'AP1S2', 'PDLIM4', 'SEMA3B', 'ATOX1'], FibMarker top5是: SPARC,ITGB1,CD74,HLA-DRA,MYL9; DEG显示为qFibro; 但FibMarker可以是eFibro,可以是iFibro;SS是ap最高
  '61': 'Mono/Macro_APOE_liver',  
  '62': 'Epi_KRT8_pan',  
  '63': 'eFibro_COL6A1_adipose', 
  '64': 'eFibro_COL1A2_pan', 
  '65': 'qFibro_MGP_eye'  # ***  改:qFibro
}  

adata.obs['Celltype'] = adata.obs['leiden'].map(cluster_to_celltype)
unmapped_clusters = set(adata.obs['leiden']) - set(cluster_to_celltype.keys())
if unmapped_clusters:
    print(f"未定义的聚类: {unmapped_clusters}") 
    adata.obs['Celltype'] = adata.obs['leiden'].map(lambda x: cluster_to_celltype.get(x, 'Unknown'))

adata.obs['Celltype_sub'] = adata.obs['Celltype'].str.split('_').str[:-1].str.join('_')
adata.obs['Celltype_general'] = adata.obs['Celltype_sub'].str.split('_').str[:-1].str.join('_')
adata.write('/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn.h5ad')



In [ ]:
####### 为了注释是 leiden 是 tissue_specific 还是 tissue_pan 的，画个 dominant tissue 的图
import pandas as pd
import scanpy as sc
import seaborn as sns
# 假设 adata 是您的单细胞对象，且包含 'leiden' 和 'tissue' 列
# 如果列名不同（如 'kissue'），请替换下方代码中的 'tissue'

# 1. 统计每个 leiden 中不同 tissue 的计数
meta_data = adata.obs
count_df = meta_data.groupby(['leiden', 'tissue']).size().unstack(fill_value=0)                 #    adata.obs.groupby(['leiden', 'tissue']).size().unstack(fill_value=0)

# 2. 计算每个 leiden 中 tissue 的占比（行内比例）
percentage_df = count_df.div(count_df.sum(axis=1), axis=0) * 100  # 转换为百分比
# 3. 提取每个 leiden 中占比最高的 tissue 及其比例
max_tissue = percentage_df.idxmax(axis=1)
max_percentage = percentage_df.max(axis=1)

# 4. 构建结果 DataFrame
dominant_tissue_percent = pd.DataFrame({
    'leiden': max_tissue.index,
    'dominant': max_tissue.values,
    'percentage': max_percentage.values.round(2)  # 保留两位小数
}).reset_index(drop=True)

# 假设 df1 是之前生成的结果，包含列：'leiden', 'dominant_tissue', 'percentage'
# 设置图形风格
sns.set_style("whitegrid")
plt.figure(figsize=(18, 8))

df1 = dominant_tissue_percent.sort_values('percentage')
leiden_order = df1.sort_values('percentage')['leiden'].tolist()
# 绘制柱状图（颜色按 dominant_tissue 自动分配）
ax = sns.barplot(
    data=df1,
    x="leiden",
    y="percentage",
    order=leiden_order,
    # color="dominant",
    hue="dominant",  # 按 tissue 类型着色
    # palette="viridis",      # 使用 viridis 调色板（可替换为 'tab20' 等）
    dodge=False            # 确保柱子不分开
)

# 添加标题和标签
plt.title("Dominant Tissue Percentage", fontsize=14)
plt.xlabel("Leiden", fontsize=12)
plt.ylabel("Percentage(%)", fontsize=12)

# 调整图例位置和样式
plt.legend(
    title="Dominant Tissue",
    bbox_to_anchor=(1, 1),  # 将图例放在图形右侧
    loc="upper left",
    frameon=False,
    fontsize=13
)

# 显示百分比标签（可选）
for p in ax.patches:
    height = p.get_height()
    ax.text(
        p.get_x() + p.get_width() / 2.,
        height + 1,
        f"{height:.1f}",
        ha="center",
        rotation=90,
        fontsize=12
    )

# 自动调整布局避免标签重叠
plt.tight_layout()
fig_dir='external_inputs/annotation'
# plt.savefig(
#     fig_dir+"Fibro_leiden.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_dir+"Dominant_tissue.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False )
plt.show()



In [ ]:
##### 画 Celltype_general 的代码 
plt.rcParams['figure.figsize'] = (7.5, 5)
fig, ax = plt.subplots()
# 关键：指定 palette=Nat_color_list
sc.pl.umap(
    adata,
    color='Celltype_general',
    # palette=Nat_color_list,
    ax=ax,
    show=False,
    s=10,
    alpha=0.8
)
ax.set_title('Celltype_general', fontsize=16, fontweight='bold', pad=20)
# ================================
# 6. 去边框
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
fig_dir='external_inputs/umap' 
# plt.savefig(
#     fig_dir+"Fibro_leiden.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf'
# )
plt.savefig(fig_dir+"Fibro_Celltype_general.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False )
plt.show()


In [ ]:
############## 之前那一版随机采样 + 评估 + 可视化 的主代码 
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import pdist
from scipy.stats import f_oneway, kruskal
import warnings
warnings.filterwarnings('ignore')

###### 核心评估指标
def evaluate_batch_integration(adata, batch_key='tissue', label_key='leiden', n_pcs=50):
    """
    评估批次整合效果的低内存消耗函数
    Parameters
    ----------
    adata : AnnData
        整合后的单细胞对象
    batch_key : str
        批次信息的列名（这里为'tissue'）
    label_key : str
        聚类标签列名
    n_pcs : int
        使用的PCA主成分数
    Returns
    -------
    dict : 包含各项评估指标的字典
    """
    print("开始评估批次整合效果...")
    results = {}
    # 1. 批次分布均衡性（最基础）
    batch_counts = adata.obs[batch_key].value_counts()
    results['n_batches'] = len(batch_counts)
    results['batch_sizes'] = batch_counts.to_dict()
    results['batch_balance_cv'] = batch_counts.std() / batch_counts.mean()  # 变异系数
    # 2. 使用PCA空间评估（低内存）
    X_pca = adata.obsm['X_pca'][:, :min(n_pcs, adata.obsm['X_pca'].shape[1])]
    # 采样策略：如果细胞数太多，随机采样进行评估
    max_cells_for_eval = 20000
    if X_pca.shape[0] > max_cells_for_eval:
        print(f"细胞数过多({X_pca.shape[0]})，随机采样{max_cells_for_eval}个细胞进行评估")
        np.random.seed(42)
        sample_idx = np.random.choice(X_pca.shape[0], max_cells_for_eval, replace=False)
        X_pca_sampled = X_pca[sample_idx]
        batch_sampled = adata.obs[batch_key].iloc[sample_idx]
        label_sampled = adata.obs[label_key].iloc[sample_idx]
    else:
        X_pca_sampled = X_pca
        batch_sampled = adata.obs[batch_key]
        label_sampled = adata.obs[label_key]
    
    # 3. 批次内 vs 批次间距离比（关键指标）
    print("计算批次内/批次间距离...")
    from sklearn.metrics.pairwise import euclidean_distances
    # 采样计算距离矩阵（避免O(n²)内存）
    sample_size_per_batch = min(500, X_pca_sampled.shape[0] // results['n_batches'])
    sampled_indices = []
    sampled_batches = []
    for batch in batch_sampled.unique():
        batch_idx = np.where(batch_sampled == batch)[0]
        if len(batch_idx) > sample_size_per_batch:
            batch_idx = np.random.choice(batch_idx, sample_size_per_batch, replace=False)
        sampled_indices.extend(batch_idx)
        sampled_batches.extend([batch] * len(batch_idx))
    
    sampled_indices = np.array(sampled_indices)
    X_sampled = X_pca_sampled[sampled_indices]
    batch_labels = np.array(sampled_batches)
    # 计算距离
    distances = euclidean_distances(X_sampled)
    # 批次内距离
    within_batch_distances = []
    between_batch_distances = []
    
    for i in range(len(X_sampled)):
        for j in range(i+1, len(X_sampled)):
            if batch_labels[i] == batch_labels[j]:
                within_batch_distances.append(distances[i, j])
            else:
                between_batch_distances.append(distances[i, j])
    
    results['within_batch_mean_dist'] = np.mean(within_batch_distances)
    results['between_batch_mean_dist'] = np.mean(between_batch_distances)
    results['batch_mixing_ratio'] = results['within_batch_mean_dist'] / results['between_batch_mean_dist']
    # 比值越接近1越好，>1表示批次内距离大于批次间（可能过度校正）
    # 4. 轮廓系数（衡量聚类紧密度）
    print("计算轮廓系数...")
    try:
        # 进一步采样以减少计算时间
        max_silhouette_samples = 10000
        if X_pca_sampled.shape[0] > max_silhouette_samples:
            sil_idx = np.random.choice(X_pca_sampled.shape[0], max_silhouette_samples, replace=False)
            X_sil = X_pca_sampled[sil_idx]
            labels_sil = label_sampled.iloc[sil_idx]
        else:
            X_sil = X_pca_sampled
            labels_sil = label_sampled
        
        sil_score = silhouette_score(X_sil, labels_sil, metric='euclidean', sample_size=min(5000, len(X_sil)))
        results['silhouette_score'] = sil_score
        # 轮廓系数 > 0.5 表示聚类良好，> 0.7 表示很强
    except Exception as e:
        print(f"轮廓系数计算失败: {e}")
        results['silhouette_score'] = np.nan
    
    # 5. 批次熵（评估批次混合程度）
    print("计算批次混合熵...")
    from sklearn.neighbors import NearestNeighbors
    n_neighbors = min(30, X_pca_sampled.shape[0] - 1)
    nn = NearestNeighbors(n_neighbors=n_neighbors, metric='euclidean')
    nn.fit(X_pca_sampled)
    distances, indices = nn.kneighbors(X_pca_sampled)
    
    # 计算每个细胞的局部批次熵
    batch_entropy_list = []
    for i in range(len(X_pca_sampled)):
        neighbor_batches = batch_sampled.iloc[indices[i]]
        batch_probs = neighbor_batches.value_counts(normalize=True)
        entropy = -sum(p * np.log2(p + 1e-10) for p in batch_probs)
        batch_entropy_list.append(entropy)
    
    results['mean_batch_entropy'] = np.mean(batch_entropy_list)
    results['max_possible_entropy'] = np.log2(results['n_batches'])
    results['normalized_entropy'] = results['mean_batch_entropy'] / results['max_possible_entropy']
    # 归一化熵越接近1，批次混合越好
    # 6. 批次特异性方差（ANOVA）
    print("计算批次间方差...")
    batch_var_explained = []
    for i in range(min(20, X_pca_sampled.shape[1])):  # 只计算前20个PC
        pc_values = X_pca_sampled[:, i]
        groups = [pc_values[batch_sampled == b] for b in batch_sampled.unique()]
        # 使用Kruskal-Wallis检验（非参数ANOVA）
        try:
            h_stat, p_val = kruskal(*groups)
            # 计算效应量（eta squared）
            ss_between = sum(len(g) * (np.mean(g) - np.mean(pc_values))**2 for g in groups)
            ss_total = sum((pc_values - np.mean(pc_values))**2)
            eta_sq = ss_between / ss_total if ss_total > 0 else 0
            batch_var_explained.append(eta_sq)
        except:
            batch_var_explained.append(0)
    
    results['mean_batch_variance'] = np.mean(batch_var_explained)
    # 值越小越好，表示批次效应越小
    # 7. 打印总结
    print("\n" + "="*60)
    print("批次整合评估结果:")
    print("="*60)
    print(f"批次数量: {results['n_batches']}")
    print(f"批次大小变异系数: {results['batch_balance_cv']:.3f}")
    print(f"批次内/批次间距离比: {results['batch_mixing_ratio']:.3f} (越接近1越好)")
    print(f"轮廓系数: {results['silhouette_score']:.3f} (>0.5表示良好)")
    print(f"归一化批次熵: {results['normalized_entropy']:.3f} (越接近1越好)")
    print(f"批次方差解释度: {results['mean_batch_variance']:.3f} (越小越好)")
    return results


##### 可视化评估代码
def visualize_integration(adata, results, batch_key='tissue', label_key='leiden', save_path=None):
    """
    可视化批次整合效果（低内存版本）
    """
    fig = plt.figure(figsize=(20, 12))
    
    # 1. UMAP按批次着色
    ax1 = fig.add_subplot(2, 3, 1)
    sc.pl.umap(adata, color=batch_key, ax=ax1, show=False, title=f'UMAP - {batch_key} (Batch)', size=2)
    
    # 2. UMAP按聚类着色
    ax2 = fig.add_subplot(2, 3, 2)
    sc.pl.umap(adata, color=label_key, ax=ax2, show=False, title=f'UMAP - {label_key} (Clusters)', size=2)
    
    # 3. 批次分布饼图
    ax3 = fig.add_subplot(2, 3, 3)
    batch_counts = adata.obs[batch_key].value_counts()
    colors = plt.cm.Set3(np.linspace(0, 1, len(batch_counts)))
    wedges, texts, autotexts = ax3.pie(batch_counts.values, labels=batch_counts.index, 
                                        autopct='%1.1f%%', colors=colors, startangle=90)
    ax3.set_title(f'Batch Distribution (n={len(batch_counts)})')
    
    # 4. 批次混合熵分布
    ax4 = fig.add_subplot(2, 3, 4)
    # 重新计算熵（采样）
    X_pca = adata.obsm['X_pca'][:, :min(30, adata.obsm['X_pca'].shape[1])]
    max_samples = 10000
    if X_pca.shape[0] > max_samples:
        sample_idx = np.random.choice(X_pca.shape[0], max_samples, replace=False)
        X_sample = X_pca[sample_idx]
        batch_sample = adata.obs[batch_key].iloc[sample_idx]
    else:
        X_sample = X_pca
        batch_sample = adata.obs[batch_key]
    
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=min(30, X_sample.shape[0]-1), metric='euclidean')
    nn.fit(X_sample)
    distances, indices = nn.kneighbors(X_sample)
    
    entropies = []
    for i in range(len(X_sample)):
        neighbor_batches = batch_sample.iloc[indices[i]]
        probs = neighbor_batches.value_counts(normalize=True)
        entropy = -sum(p * np.log2(p + 1e-10) for p in probs)
        entropies.append(entropy)
    
    ax4.hist(entropies, bins=30, alpha=0.7, color='steelblue', edgecolor='black')
    ax4.axvline(results['mean_batch_entropy'], color='red', linestyle='--', 
                label=f"Mean: {results['mean_batch_entropy']:.2f}")
    ax4.set_xlabel('Batch Entropy (bits)')
    ax4.set_ylabel('Frequency')
    ax4.set_title(f'Batch Mixing Entropy Distribution\nNormalized: {results["normalized_entropy"]:.3f}')
    ax4.legend()
    
    # 5. 批次距离比条形图
    ax5 = fig.add_subplot(2, 3, 5)
    metrics = ['Within-batch\nDistance', 'Between-batch\nDistance']
    values = [results['within_batch_mean_dist'], results['between_batch_mean_dist']]
    bars = ax5.bar(metrics, values, color=['#FF6B6B', '#4ECDC4'])
    ax5.set_ylabel('Mean Euclidean Distance')
    ax5.set_title(f'Batch Distance Ratio: {results["batch_mixing_ratio"]:.3f}')
    # 添加数值标签
    for bar, val in zip(bars, values):
        ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)
    
    # 6. 指标雷达图
    ax6 = fig.add_subplot(2, 3, 6, projection='polar')
    # 归一化指标（0-1范围，1表示最好）
    normalized_metrics = {
        'Batch Mixing\nRatio': min(1.0, results['batch_mixing_ratio']),
        'Silhouette\nScore': min(1.0, max(0, results['silhouette_score'] / 1.0)),
        'Batch\nEntropy': results['normalized_entropy'],
        'Batch\nVariance': 1 - min(1.0, results['mean_batch_variance'])
    }
    
    categories = list(normalized_metrics.keys())
    values = list(normalized_metrics.values())
    values += values[:1]  # 闭合图形
    angles = [n / float(len(categories)) * 2 * np.pi for n in range(len(categories))]
    angles += angles[:1]
    
    ax6.plot(angles, values, 'o-', linewidth=2, color='darkblue')
    ax6.fill(angles, values, alpha=0.25, color='steelblue')
    ax6.set_xticks(angles[:-1])
    ax6.set_xticklabels(categories, size=8)
    ax6.set_ylim(0, 1)
    ax6.set_title('Integration Quality Metrics\n(1=Best)', size=10, pad=20)
    ax6.grid(True)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"图表已保存至: {save_path}")
    plt.show()
    
    return fig


def compare_before_after_integration(adata_raw, adata_integrated, batch_key='tissue', 
                                     save_path=None):
    """
    对比整合前后的批次效应（需要原始未整合的数据）
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    # 整合前
    ax1 = axes[0]
    sc.pl.umap(adata_raw, color=batch_key, ax=ax1, show=False, 
               title=f'Before Integration - {batch_key}', size=2)
    # 整合后
    ax2 = axes[1]
    sc.pl.umap(adata_integrated, color=batch_key, ax=ax2, show=False, 
               title=f'After BBKNN Integration - {batch_key}', size=2)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"对比图已保存至: {save_path}")
    plt.show()




In [ ]:
####### 展示下 Celltype, Celltype_general, Celltype_sub 的 UMAP图
sc.pl.umap(adata, color=['tissue','leiden','Celltype_general', 'Celltype_sub', 'Celltype'], show=True, ncols =2, wspace=0.4 , hspace=0.5)  # wspace=0.3,legend_loc ='on data'
fig_dir='external_inputs/OnlyFibro'
plt.savefig(fig_dir+"/Fib_Celltype_umap.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False)
plt.show()



In [ ]:
# ==================== 主程序 ====================
# 2. 评估整合效果
results = evaluate_batch_integration(
    adata, 
    batch_key='tissue',  # 你使用的批次键
    label_key='leiden',   # 聚类标签
    n_pcs=30
)

# 3. 可视化结果
fig = visualize_integration(
    adata, 
    results, 
    batch_key='tissue',
    label_key='leiden',
    save_path='external_inputs/batch_integration_evaluation.png'
)

# 4. 保存评估结果到文件
results_df = pd.DataFrame([results])
results_df.to_csv('external_inputs/batch_integration_metrics.csv', index=False)
print("评估指标已保存")

# 5. 输出可接受性判断
print("\n" + "="*60)
print("批次整合效果可接受性判断:")
print("="*60)

# 判断标准（根据经验设定）
criteria = {
    'batch_mixing_ratio': (0.8, 1.2),  # 在0.8-1.2之间较好
    'silhouette_score': 0.3,            # >0.3可接受，>0.5良好
    'normalized_entropy': 0.6,          # >0.6表示混合良好
    'mean_batch_variance': 0.1          # <0.1表示批次效应小
}

acceptance = True
if results['batch_mixing_ratio'] < criteria['batch_mixing_ratio'][0] or results['batch_mixing_ratio'] > criteria['batch_mixing_ratio'][1]:
    print(f"⚠️  批次距离比 {results['batch_mixing_ratio']:.3f} 超出理想范围 {criteria['batch_mixing_ratio']}")
    acceptance = False
else:
    print(f"✓ 批次距离比 {results['batch_mixing_ratio']:.3f} 在理想范围内")

if results['silhouette_score'] < criteria['silhouette_score']:
    print(f"⚠️  轮廓系数 {results['silhouette_score']:.3f} 低于阈值 {criteria['silhouette_score']}")
    acceptance = False
else:
    print(f"✓ 轮廓系数 {results['silhouette_score']:.3f} 可接受")

if results['normalized_entropy'] < criteria['normalized_entropy']:
    print(f"⚠️  归一化熵 {results['normalized_entropy']:.3f} 低于阈值 {criteria['normalized_entropy']}")
    acceptance = False
else:
    print(f"✓ 归一化熵 {results['normalized_entropy']:.3f} 可接受")

if results['mean_batch_variance'] > criteria['mean_batch_variance']:
    print(f"⚠️  批次方差 {results['mean_batch_variance']:.3f} 高于阈值 {criteria['mean_batch_variance']}")
    acceptance = False
else:
    print(f"✓ 批次方差 {results['mean_batch_variance']:.3f} 可接受")

if acceptance:
    print("\n✓✓✓ 总体评估：批次整合效果可接受！ ✓✓✓")
    print("以 tissue 为批次键的整合方案在当前内存限制下是可行的。")
else:
    print("\n✗✗✗ 总体评估：批次整合效果需改进 ✗✗✗")
    print("建议考虑：")
    print("1. 调整 BBKNN 参数（neighbors_within_batch, n_pcs）")
    print("2. 考虑使用更保守的整合策略")
    print("3. 在特定细胞类型中单独分析批次效应")



In [ ]:
####### 新的一版取样方式，考虑 Leiden & Celltype
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from scipy.stats import kruskal
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

def stratified_sampling(adata, label_key='leiden', target_total=30000, min_cells_per_cluster=50):
    """
    分层采样：按 Leiden 聚类保持原始比例进行采样
    Parameters
    ----------
    adata : AnnData
        单细胞对象
    label_key : str
        聚类标签列名
    target_total : int
        目标采样总数
    min_cells_per_cluster : int
        每个聚类最少采样细胞数
    Returns
    -------
    sample_idx : array
        采样细胞的索引
    sampling_proportions : dict
        每个聚类的采样比例
    """
    print(f"\n开始分层采样，目标总数: {target_total}")
    cluster_counts = adata.obs[label_key].value_counts()
    total_cells = len(adata)
    # 计算每个聚类的目标采样数（按比例）
    sampling_proportions = {}
    sample_sizes = {}
    for cluster, count in cluster_counts.items():
        proportion = count / total_cells
        target_size = max(min_cells_per_cluster, int(target_total * proportion))
        # 不能超过该聚类的实际细胞数
        target_size = min(target_size, count)
        sample_sizes[cluster] = target_size
        sampling_proportions[cluster] = target_size / count
        print(f"  Cluster {cluster}: {count} cells -> sample {target_size} cells "
              f"(比例: {proportion:.2%}, 采样率: {target_size/count:.1%})")
    # 调整总数（因为 min_cells_per_cluster 可能导致总数超过 target_total）
    actual_total = sum(sample_sizes.values())
    if actual_total > target_total * 1.1:  # 超过10%需要调整
        print(f"\n调整采样数（当前总数: {actual_total}，目标: {target_total}）")
        scale = target_total / actual_total
        for cluster in sample_sizes:
            sample_sizes[cluster] = max(min_cells_per_cluster, int(sample_sizes[cluster] * scale))
            sample_sizes[cluster] = min(sample_sizes[cluster], cluster_counts[cluster])
    # 执行采样
    sample_idx = []
    np.random.seed(42)
    for cluster, size in sample_sizes.items():
        cluster_idx = np.where(adata.obs[label_key] == cluster)[0]
        if len(cluster_idx) > size:
            sampled = np.random.choice(cluster_idx, size, replace=False)
        else:
            sampled = cluster_idx
        sample_idx.extend(sampled)
    sample_idx = np.array(sample_idx)
    print(f"\n实际采样总数: {len(sample_idx)}")
    return sample_idx, sampling_proportions




def evaluate_batch_integration_stratified(adata, batch_key='tissue', label_key='leiden', 
                                           n_pcs=50, target_samples=30000):
    """
    使用分层采样评估批次整合效果（低内存消耗）
    Parameters
    ----------
    adata : AnnData
        整合后的单细胞对象
    batch_key : str
        批次信息的列名
    label_key : str
        聚类标签列名
    n_pcs : int
        使用的PCA主成分数
    target_samples : int
        目标采样细胞数（200万细胞采样3万，约1.5%）
    Returns
    -------
    results : dict
        包含各项评估指标的字典
    sample_idx : array
        采样细胞的索引
    """
    print("="*60)
    print("开始评估批次整合效果（分层采样模式）")
    print("="*60)
    print(f"原始数据: {adata.shape[0]} 细胞, {adata.shape[1]} 基因")
    # 1. 分层采样
    sample_idx, sampling_props = stratified_sampling(
        adata, 
        label_key=label_key, 
        target_total=target_samples,
        min_cells_per_cluster=50
    )
    # 2. 创建采样子集
    adata_sub = adata[sample_idx, :].copy()
    print(f"\n采样子集: {adata_sub.shape[0]} 细胞")
    # 3. 获取PCA空间
    X_pca = adata_sub.obsm['X_pca'][:, :min(n_pcs, adata_sub.obsm['X_pca'].shape[1])]
    batch_labels = adata_sub.obs[batch_key].values
    cluster_labels = adata_sub.obs[label_key].values
    results = {}
    results['sampling_info'] = {
        'original_cells': adata.shape[0],
        'sampled_cells': len(sample_idx),
        'sampling_rate': len(sample_idx) / adata.shape[0],
        'clusters_sampled': len(adata_sub.obs[label_key].unique())
    }
    # 4. 批次分布统计
    batch_counts = adata_sub.obs[batch_key].value_counts()
    results['n_batches'] = len(batch_counts)
    results['batch_sizes'] = batch_counts.to_dict()
    results['batch_balance_cv'] = batch_counts.std() / batch_counts.mean() if batch_counts.mean() > 0 else 0
    # 5. 计算批次内/批次间距离（优化版）
    print("\n计算批次内/批次间距离...")
    from sklearn.metrics.pairwise import euclidean_distances
    # 每个批次进一步采样（用于距离计算，避免O(n²)）
    max_per_batch_for_dist = min(800, X_pca.shape[0] // results['n_batches'])
    dist_sample_idx = []
    dist_batch_labels = []
    for batch in adata_sub.obs[batch_key].unique():
        batch_mask = adata_sub.obs[batch_key] == batch
        batch_indices = np.where(batch_mask)[0]
        if len(batch_indices) > max_per_batch_for_dist:
            sampled = np.random.choice(batch_indices, max_per_batch_for_dist, replace=False)
        else:
            sampled = batch_indices
        dist_sample_idx.extend(sampled)
        dist_batch_labels.extend([batch] * len(sampled))
    
    dist_sample_idx = np.array(dist_sample_idx)
    X_dist = X_pca[dist_sample_idx]
    batch_dist_labels = np.array(dist_batch_labels)
    
    # 计算距离矩阵（限制大小）
    n_dist_samples = len(X_dist)
    if n_dist_samples < 2000:  # 如果样本数少，直接计算全矩阵
        distances = euclidean_distances(X_dist)
        within_distances = []
        between_distances = []
        for i in range(n_dist_samples):
            for j in range(i+1, n_dist_samples):
                if batch_dist_labels[i] == batch_dist_labels[j]:
                    within_distances.append(distances[i, j])
                else:
                    between_distances.append(distances[i, j])
    else:
        # 采样计算距离对
        n_pairs = min(50000, n_dist_samples * (n_dist_samples - 1) // 2)
        within_distances = []
        between_distances = []
        
        for _ in range(n_pairs):
            i, j = np.random.choice(n_dist_samples, 2, replace=False)
            dist = np.linalg.norm(X_dist[i] - X_dist[j])
            if batch_dist_labels[i] == batch_dist_labels[j]:
                within_distances.append(dist)
            else:
                between_distances.append(dist)
    
    results['within_batch_mean_dist'] = np.mean(within_distances) if within_distances else 0
    results['between_batch_mean_dist'] = np.mean(between_distances) if between_distances else 0
    results['batch_mixing_ratio'] = (results['within_batch_mean_dist'] / 
                                      results['between_batch_mean_dist']) if results['between_batch_mean_dist'] > 0 else 1
    
    # 6. 轮廓系数
    print("计算轮廓系数...")
    try:
        max_silhouette = min(15000, X_pca.shape[0])
        if X_pca.shape[0] > max_silhouette:
            sil_idx = np.random.choice(X_pca.shape[0], max_silhouette, replace=False)
            X_sil = X_pca[sil_idx]
            labels_sil = cluster_labels[sil_idx]
        else:
            X_sil = X_pca
            labels_sil = cluster_labels
        
        sil_score = silhouette_score(X_sil, labels_sil, metric='euclidean', 
                                      sample_size=min(5000, len(X_sil)))
        results['silhouette_score'] = sil_score
    except Exception as e:
        print(f"轮廓系数计算失败: {e}")
        results['silhouette_score'] = np.nan
    
    # 7. 批次混合熵
    print("计算批次混合熵...")
    nn = NearestNeighbors(n_neighbors=min(30, X_pca.shape[0]-1), metric='euclidean', n_jobs=-1)
    nn.fit(X_pca)
    distances, indices = nn.kneighbors(X_pca)
    
    entropies = []
    for i in range(X_pca.shape[0]):
        neighbor_batches = batch_labels[indices[i]]
        unique_batches, counts = np.unique(neighbor_batches, return_counts=True)
        probs = counts / len(neighbor_batches)
        entropy = -np.sum(p * np.log2(p + 1e-10) for p in probs)
        entropies.append(entropy)
    
    results['mean_batch_entropy'] = np.mean(entropies)
    results['max_possible_entropy'] = np.log2(results['n_batches'])
    results['normalized_entropy'] = (results['mean_batch_entropy'] / 
                                      results['max_possible_entropy']) if results['max_possible_entropy'] > 0 else 0
    
    # 8. 批次特异性方差
    print("计算批次间方差...")
    batch_var_explained = []
    n_pcs_to_test = min(20, X_pca.shape[1])
    
    for i in range(n_pcs_to_test):
        pc_values = X_pca[:, i]
        groups = [pc_values[batch_labels == b] for b in np.unique(batch_labels)]
        groups = [g for g in groups if len(g) > 0]
        
        if len(groups) >= 2:
            try:
                h_stat, p_val = kruskal(*groups)
                # 计算效应量
                all_values = np.concatenate(groups)
                ss_between = sum(len(g) * (np.mean(g) - np.mean(all_values))**2 for g in groups)
                ss_total = np.sum((all_values - np.mean(all_values))**2)
                eta_sq = ss_between / ss_total if ss_total > 0 else 0
                batch_var_explained.append(eta_sq)
            except:
                batch_var_explained.append(0)
    
    results['mean_batch_variance'] = np.mean(batch_var_explained) if batch_var_explained else 0
    
    # 9. 每个聚类的批次组成（评估批次混合质量）
    print("计算每个聚类的批次组成...")
    cluster_batch_composition = {}
    for cluster in np.unique(cluster_labels):
        cluster_mask = cluster_labels == cluster
        cluster_batches = batch_labels[cluster_mask]
        batch_counts_cluster = pd.Series(cluster_batches).value_counts(normalize=True)
        cluster_batch_composition[cluster] = batch_counts_cluster.to_dict()
        
        # 计算该聚类的批次熵
        probs = batch_counts_cluster.values
        entropy = -np.sum(p * np.log2(p + 1e-10) for p in probs)
        cluster_batch_composition[cluster]['entropy'] = entropy
        cluster_batch_composition[cluster]['normalized_entropy'] = entropy / np.log2(results['n_batches'])
    
    results['cluster_batch_composition'] = cluster_batch_composition
    results['mean_cluster_batch_entropy'] = np.mean([v['normalized_entropy'] 
                                                      for v in cluster_batch_composition.values()])
    
    # 10. 打印总结
    print("\n" + "="*60)
    print("批次整合评估结果:")
    print("="*60)
    print(f"采样信息: {results['sampling_info']['sampled_cells']}/{results['sampling_info']['original_cells']} "
          f"({results['sampling_info']['sampling_rate']:.1%})")
    print(f"批次数量: {results['n_batches']}")
    print(f"批次大小变异系数: {results['batch_balance_cv']:.3f}")
    print(f"批次内/批次间距离比: {results['batch_mixing_ratio']:.3f} (越接近1越好)")
    print(f"轮廓系数: {results['silhouette_score']:.3f} (>0.5良好)")
    print(f"归一化批次熵: {results['normalized_entropy']:.3f} (越接近1越好)")
    print(f"批次方差解释度: {results['mean_batch_variance']:.3f} (越小越好)")
    print(f"平均聚类内批次熵: {results['mean_cluster_batch_entropy']:.3f} (>0.6表示聚类内批次混合好)")
    return results, sample_idx, adata_sub

    



In [ ]:
################ 对 metrics进行可视化
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import scanpy as sc
import numpy as np
from sklearn.neighbors import NearestNeighbors

def visualize_integration_stratified(adata_sub, results, batch_key='tissue', 
                                      label_key='leiden', save_path=None):
    """
    基于采样数据的可视化（保持原始分布特征）
    修改：调整UMAP图例位置，优化子图比例
    """
    # 调整整体图形尺寸，增加宽度以容纳图例
    fig = plt.figure(figsize=(16, 10))  # 宽度从22增加到24，高度从14降到12使第二行更细长
    # 调整GridSpec的行高比例，让第二行相对更细长
    gs = GridSpec(nrows=2, ncols=4, figure=fig, hspace=0.2, wspace=1,
                  height_ratios=[1.2, 0.8])  # 第一行稍高，第二行更细长
    
    # 1. UMAP按批次着色（使用采样数据）- 调整图例位置
    ax1 = fig.add_subplot(gs[0, 0:2])
    # 绘制UMAP，将图例放在右侧外部
    sc.pl.umap(adata_sub, color=batch_key, ax=ax1, show=False, 
               title=f'UMAP - {batch_key} (Sampled, n={adata_sub.shape[0]})', 
               size=15, alpha=0.7,
               legend_loc='right margin',  # 图例放在右侧边缘
               colorbar_loc=None)  # 不显示颜色条
    
    # 2. UMAP按聚类着色 - 调整图例位置和多列显示
    ax2 = fig.add_subplot(gs[0, 2:4])
    # 获取唯一的聚类标签数量，决定图例列数
    n_clusters = len(adata_sub.obs[label_key].unique())
    # ncol_legend = min(3, max(1, n_clusters // 10))  # 根据聚类数动态调整图例列数
    
    sc.pl.umap(adata_sub, color=label_key, ax=ax2, show=False, 
               title=f'UMAP - {label_key} (Sampled)', size=15, alpha=0.7,
               legend_loc='right margin',  # 图例放在右侧边缘
               colorbar_loc=None,
               legend_fontsize=8  # 减小图例字体
               )  # 多列显示图例
    
    # 3. 批次分布条形图 - 调整条形宽度
    ax3 = fig.add_subplot(gs[1, 0])
    batch_counts = adata_sub.obs[batch_key].value_counts()
    colors = plt.cm.Set3(np.linspace(0, 1, len(batch_counts)))
    
    # 设置更细的条形宽度
    x_pos = np.arange(len(batch_counts))
    bars = ax3.bar(x_pos, batch_counts.values, color=colors, width=0.6, edgecolor='black')
    
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(batch_counts.index, rotation=45, ha='right', fontsize=8)
    ax3.set_ylabel('Number of Cells', fontsize=9)
    ax3.set_title(f'Batch Distribution\nCV={results["batch_balance_cv"]:.3f}', fontsize=10)
    
    # 添加数值标签（调整位置）
    for bar, val in zip(bars, batch_counts.values):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(batch_counts.values)*0.01, 
                str(val), ha='center', va='bottom', fontsize=7)
    
    # 调整y轴范围，给标签留出空间
    ax3.set_ylim(0, max(batch_counts.values) * 1.1)
    
    # 4. 批次混合熵分布 - 调整图形比例
    ax4 = fig.add_subplot(gs[1, 1])
    # 重新计算熵（使用采样子集）
    X_pca = adata_sub.obsm['X_pca'][:, :min(30, adata_sub.obsm['X_pca'].shape[1])]
    nn = NearestNeighbors(n_neighbors=min(30, X_pca.shape[0]-1), metric='euclidean', n_jobs=-1)
    nn.fit(X_pca)
    distances, indices = nn.kneighbors(X_pca)
    
    entropies = []
    for i in range(X_pca.shape[0]):
        neighbor_batches = adata_sub.obs[batch_key].iloc[indices[i]]
        probs = neighbor_batches.value_counts(normalize=True)
        entropy = -sum(p * np.log2(p + 1e-10) for p in probs)
        entropies.append(entropy)
    
    # 使用更细的直方图柱子
    ax4.hist(entropies, bins=40, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)
    ax4.axvline(results['mean_batch_entropy'], color='red', linestyle='--', 
                linewidth=1.5, label=f"Mean: {results['mean_batch_entropy']:.2f}")
    ax4.axvline(results['max_possible_entropy'], color='green', linestyle=':', 
                linewidth=1.5, label=f"Max possible: {results['max_possible_entropy']:.2f}")
    ax4.set_xlabel('Batch Entropy (bits)', fontsize=9)
    ax4.set_ylabel('Frequency', fontsize=9)
    ax4.set_title(f'Batch Mixing Entropy\nNormalized: {results["normalized_entropy"]:.3f}', fontsize=10)
    ax4.legend(fontsize=7, loc='upper left')
    ax4.tick_params(labelsize=8)
    
    # 5. 批次距离比 - 使柱子更细
    ax5 = fig.add_subplot(gs[1, 2])
    metrics = ['Within-batch\nDistance', 'Between-batch\nDistance']
    values = [results['within_batch_mean_dist'], results['between_batch_mean_dist']]
    
    # 使用更细的条形
    x_pos = np.arange(len(metrics))
    bars = ax5.bar(x_pos, values, width=0.5, color=['#FF6B6B', '#4ECDC4'], edgecolor='black')
    
    ax5.set_ylabel('Euclidean Distance (mean)', fontsize=9)
    ax5.set_title(f'Batch Distance Ratio: {results["batch_mixing_ratio"]:.3f}', fontsize=10)
    ax5.set_xticks(x_pos)
    ax5.set_xticklabels(metrics, fontsize=8)
    
    # 添加数值标签
    for bar, val in zip(bars, values):
        ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.02, 
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    
    # 调整y轴范围
    ax5.set_ylim(0, max(values) * 1.15)
    ax5.tick_params(labelsize=8)
    
    # 6. 聚类内批次混合热图 - 调整热图比例
    ax6 = fig.add_subplot(gs[1, 3])
    # 构建热图数据
    clusters = list(results['cluster_batch_composition'].keys())
    batches = list(results['batch_sizes'].keys())
    
    # 创建矩阵
    heatmap_data = np.zeros((len(clusters), len(batches)))
    for i, cluster in enumerate(clusters):
        comp = results['cluster_batch_composition'][cluster]
        for j, batch in enumerate(batches):
            heatmap_data[i, j] = comp.get(batch, 0)
    
    # 调整热图尺寸，使其更细长
    im = ax6.imshow(heatmap_data, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
    
    # 设置标签
    ax6.set_xticks(range(len(batches)))
    ax6.set_xticklabels(batches, rotation=45, ha='right', fontsize=7)
    ax6.set_yticks(range(len(clusters)))
    ax6.set_yticklabels(clusters, fontsize=7)
    ax6.set_xlabel('Batch', fontsize=9)
    ax6.set_ylabel('Leiden Cluster', fontsize=9)
    ax6.set_title('Batch Composition per Leiden\n(Darker = Higher Proportion)', fontsize=10)
    
    # 添加颜色条（调整大小和位置）
    cbar = plt.colorbar(im, ax=ax6, fraction=0.046, pad=0.04, aspect=30)
    cbar.ax.tick_params(labelsize=7)
    
    # 整体调整布局，为右侧图例留出空间
    plt.tight_layout()
    plt.subplots_adjust(right=0.95, left=0.05)  # 为右侧图例留出空间
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"\n图表已保存至: {save_path}")
    
    plt.show()
    return fig


def plot_quality_radar(results, save_path=None):
    """
    绘制整合质量雷达图（数值标签在轴外侧）
    """
    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(projection='polar'))
    # 归一化指标（0-1范围，1表示最好）
    normalized_metrics = {
        'Batch Mix\nRatio': min(1.0, max(0, 1 - abs(results['batch_mixing_ratio'] - 1))),
        'Silhouette\nScore': min(1.0, max(0, results['silhouette_score'] / 0.7)),
        'Batch\nEntropy': results['normalized_entropy'],
        'Batch\nVariance': 1 - min(1.0, results['mean_batch_variance']),
        'Cluster Batch\nEntropy': results['mean_cluster_batch_entropy']
    }
    categories = list(normalized_metrics.keys())
    values = list(normalized_metrics.values())
    original_values = values.copy()
    values += values[:1]  # 闭合图形
    angles = [n / float(len(categories)) * 2 * np.pi for n in range(len(categories))]
    angles += angles[:1]
    # 绘制雷达图
    ax.plot(angles, values, 'o-', linewidth=2, color='darkblue', markersize=8)
    ax.fill(angles, values, alpha=0.25, color='steelblue')
    # 设置刻度标签
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=10)
    ax.set_ylim(0, 1.15)  # 扩大y轴范围，为外侧标签留空间
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=8)
    ax.set_title('Integration Quality Metrics\n(1=Best)', size=12, pad=20)
    ax.grid(True)
    # 添加参考线
    for i, angle in enumerate(angles[:-1]):
        ax.plot([angle, angle], [0, 1], 'gray', linewidth=0.5, alpha=0.3)
    
    # ========== 在每个轴上添加数值标签（外侧） ==========
    for i, (category, value) in enumerate(zip(categories, original_values)):
        angle = angles[i]
        # 标签放在半径1.05的位置（外侧）
        r_position = 1.08
        # 根据角度调整文本对齐方式
        if angle < np.pi/6 or angle > 11*np.pi/6:
            ha = 'center'
            va = 'bottom'
        elif angle < np.pi/2:
            ha = 'left'
            va = 'center'
        elif angle < np.pi:
            ha = 'left'
            va = 'top'
        elif angle < 3*np.pi/2:
            ha = 'right'
            va = 'top'
        else:
            ha = 'right'
            va = 'center'
        
        # 添加数值标签
        ax.text(angle, r_position, f'{value:.3f}', 
                ha=ha, va=va, fontsize=9, fontweight='bold',
                bbox=dict(boxstyle="round,pad=0.2", facecolor='white', alpha=0.8, edgecolor='gray'))
    
    # 可选：在中心添加综合得分
    overall_score = np.mean(original_values)
    ax.text(0, 0, f'Overall\n{overall_score:.3f}', 
            ha='center', va='center', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle="circle,pad=0.3", facecolor='lightyellow', alpha=0.8, edgecolor='orange'))
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"雷达图已保存至: {save_path}")
    
    plt.show()
    return fig
    

In [ ]:
############# 执行主程序 
# ==================== 主程序 ====================
# 2. 评估整合效果（分层采样）
results, sample_idx, adata_sub = evaluate_batch_integration_stratified(
    adata, 
    batch_key='tissue',      # 你使用的批次键
    label_key='leiden',       # 聚类标签
    n_pcs=30,                 # 使用的PC数, 整合的时候用的也是 30，这里保持一致
    target_samples=40000      # 目标采样数（可根据需要调整）
)

# 3. 可视化结果
fig1 = visualize_integration_stratified(
    adata_sub, 
    results, 
    batch_key='tissue',
    label_key='leiden',
    save_path='external_inputs/Fibro_tissue_inte_evaluation.png'
)

# 4. 绘制雷达图
fig2 = plot_quality_radar(
    results,
    save_path='external_inputs/Fibro_tissue_inte_eval_radar.png'
)

# 5. 保存详细结果
# 保存采样索引（供后续分析使用）
# np.save('external_inputs/Endo_ds_idx_3w.npy', sample_idx)

# 保存评估指标到文件
results_summary = {
    'sampling_rate': results['sampling_info']['sampling_rate'],
    'n_batches': results['n_batches'],
    'batch_balance_cv': results['batch_balance_cv'],
    'batch_mixing_ratio': results['batch_mixing_ratio'],
    'silhouette_score': results['silhouette_score'],
    'normalized_entropy': results['normalized_entropy'],
    'mean_batch_variance': results['mean_batch_variance'],
    'mean_cluster_batch_entropy': results['mean_cluster_batch_entropy']
}

results_df = pd.DataFrame([results_summary])
results_df.to_csv('external_inputs/Fibro_tissue_inte_evaluation.csv', index=False)
print("\n评估指标已保存到 CSV 文件")

# 保存每个聚类的批次组成
cluster_composition_df = pd.DataFrame([
    {'cluster': k, **v} for k, v in results['cluster_batch_composition'].items()
])
cluster_composition_df.to_csv('external_inputs/leiden_tissueB_composition.csv', index=False)
print("聚类批次组成已保存")

# 6. 输出可接受性判断
print("\n" + "="*60)
print("批次整合效果可接受性判断:")
print("="*60)

# 判断标准
criteria = {
    'batch_mixing_ratio': (0.7, 1.3),  # 允许更宽的范围
    'silhouette_score': 0.25,           # 降低阈值适应大数据集
    'normalized_entropy': 0.5,          # 批次混合的宽松标准
    'mean_batch_variance': 0.15,        # 批次效应方差阈值
    'mean_cluster_batch_entropy': 0.5   # 聚类内批次混合
}

acceptance = True
issues = []
if results['batch_mixing_ratio'] < criteria['batch_mixing_ratio'][0] or results['batch_mixing_ratio'] > criteria['batch_mixing_ratio'][1]:
    issues.append(f"批次距离比 {results['batch_mixing_ratio']:.3f} 超出范围 {criteria['batch_mixing_ratio']}")
    acceptance = False

if results['silhouette_score'] < criteria['silhouette_score']:
    issues.append(f"轮廓系数 {results['silhouette_score']:.3f} 低于 {criteria['silhouette_score']}")

if results['normalized_entropy'] < criteria['normalized_entropy']:
    issues.append(f"归一化熵 {results['normalized_entropy']:.3f} 低于 {criteria['normalized_entropy']}")

if results['mean_batch_variance'] > criteria['mean_batch_variance']:
    issues.append(f"批次方差 {results['mean_batch_variance']:.3f} 高于 {criteria['mean_batch_variance']}")

if results['mean_cluster_batch_entropy'] < criteria['mean_cluster_batch_entropy']:
    issues.append(f"聚类内批次熵 {results['mean_cluster_batch_entropy']:.3f} 低于 {criteria['mean_cluster_batch_entropy']}")

if issues:
    print("\n发现以下问题:")
    for issue in issues:
        print(f"  ⚠️  {issue}")
    print("\n✗✗✗ 总体评估：批次整合效果需改进 ✗✗✗")
else:
    print("\n✓✓✓ 总体评估：批次整合效果可接受！ ✓✓✓")
    print("以 tissue 为批次键的整合方案在当前内存限制下是可行的。")

# 7. 输出详细建议
print("\n" + "="*60)
print("详细分析与建议:")
print("="*60)

if results['batch_mixing_ratio'] < 0.7:
    print("• 批次内距离显著小于批次间距离 → 整合不足，批次效应仍存在")
    print("  建议: 增加 neighbors_within_batch 参数或尝试其他整合方法")
elif results['batch_mixing_ratio'] > 1.3:
    print("• 批次内距离大于批次间距离 → 可能过度校正")
    print("  建议: 减少 neighbors_within_batch 参数")
else:
    print("• 批次距离比在理想范围内 ✓")

if results['normalized_entropy'] < 0.5:
    print("• 批次混合程度较低 → 不同 tissue 的细胞在局部区域分离")
    print("  建议: 考虑是否需要更细粒度的整合")
elif results['normalized_entropy'] < 0.7:
    print("• 批次混合程度中等 → 可接受，但仍有改进空间")

if results['mean_cluster_batch_entropy'] > 0.6:
    print("• 每个聚类内各批次混合良好 ✓")
elif results['mean_cluster_batch_entropy'] > 0.4:
    print("• 部分聚类存在批次偏向，需注意下游分析时批次的影响")

print("\n最终建议:")
if acceptance:
    print("✓ 当前整合方案可以接受，建议继续进行下游分析")
    print("✓ 在差异表达分析时，仍建议将 tissue 作为协变量考虑")
else:
    print("✗ 建议考虑以下改进方案:")
    print("  1. 调整 BBKNN 参数: neighbors_within_batch=5-10")
    print("  2. 增加 n_pcs 到 50-80")
    print("  3. 在主要细胞类型内部分别评估批次效应")
    print("  4. 考虑使用 Harmony 或 scVI 作为替代方案")
    




In [ ]:
########  提取只有Age信息的healthy数据拿来做training
# 将 age 转成 float，用来记录哪些有 Age信息
def isstr(x):
    try: 
        float(x)
        return False
    except ValueError as e:
        return True


adata.obs["age"] = adata.obs["age"].astype(str)
##### 下面这个函数Endo 和 Fibro不同，需要根据具体的age的unique值来进行修改
def assign_age_group(age):
    # age = str(age).lower()
    if 'hildren' in age:
        return 'B'
    elif 'nfant' in age:
        return 'A'
    elif 'Embr' in age:
        return 'A'
    elif 'Adult' in age:
        return 'C'
    elif age == 'Aged':
        return 'H'
    elif 'month' in age:
        return 'A'
    try:
        age = float(age)  # 转换为浮动类型
        if 0<=age<20:
            return 'A'
        elif 20 <= age < 30:
            return 'B'
        elif 30 <= age < 40:
            return 'C'
        elif 40 <= age < 50:
            return 'D'
        elif 50 <= age < 60:
            return 'E'
        elif 60 <= age < 70:
            return 'F'
        elif 70 <= age < 90:
            return 'G'
        elif age >= 90:
            return 'H'
    except ValueError:
        return 'Unknown'  # 对age 信息重新进行分群 



In [ ]:
adata.obs['Age_group']  = adata.obs['age'].apply(assign_age_group)
adata = adata[adata.obs['Celltype_general'].isin(['eFibro', 'Myfibro', 'qFibro', 'iFibro'])]
######## 需要对筛选出来的数据进行 leiden rename，因为Leiden可能不按顺序来了
leiden_unique = adata.obs['leiden'].unique()
leiden_numeric = np.sort([int(x) for x in leiden_unique]).tolist()  # 转为整数并排序
leiden_new = np.arange(0, len(leiden_numeric)).tolist()
if len(leiden_numeric) != len(leiden_new):
    raise ValueError("l1 和 l2 的长度必须相同")

old_to_new = {str(key): str(value) for key, value in zip(leiden_numeric, leiden_new)}
adata.obs['leiden'] = adata.obs['leiden'].map(old_to_new)


adata.write('/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyFib.h5ad')


In [ ]:
leiden_unique = fibro_degs['leiden'].unique()
leiden_numeric = np.sort([int(x) for x in leiden_unique]).tolist()  # 转为整数并排序
leiden_new = np.arange(0, len(leiden_numeric)).tolist()
if len(leiden_numeric) != len(leiden_new):
    raise ValueError("l1 和 l2 的长度必须相同")

old_to_new = {str(key): str(value) for key, value in zip(leiden_numeric, leiden_new)}
fibro_degs['leiden'] = fibro_degs['leiden'].map(old_to_new)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

Nat_color_list = ['#E64B35',  # Nature红
        '#4DBBD5',  # Nature蓝
        '#00A087',  # Nature绿
        '#3C5488',  # Nature深蓝
        '#F39B7F',  # 珊瑚色
        '#8491B4',  # 淡紫色
        '#91D1C2',  # 薄荷绿
        '#DC0000',  # 正红
        '#7E6148',  # 棕色
        '#B09C85',  # 米色
        '#F4649D',  # 粉红
        '#7C6A9C',  # 紫罗兰
        '#D9A428',  # 金黄色
        '#59A14F',  # 翠绿
        '#AF7AC4',  # 浅紫
        '#FFBB78',  # 杏色
        '#98DF8A',  # 嫩绿
        '#F7B6D2',  # 浅粉
        '#C5B0D5',  # 淡紫
        '#FF9896',  # 浅红
        '#9467BD',  # 深紫
        '#8C564B',  # 赭石
        '#E377C2',  # 玫红
        '#7F7F7F',  # 中灰
        '#BCBD22',  # 黄绿
        '#17BECF',  # 青绿
        '#AEC7E8',  # 天蓝
        '#C49C94',  # 肉色
        '#DBDB8D',  # 淡黄
        '#9EDAE5',  # 浅青
        '#F5B3A5',  # 浅珊瑚
        '#B3B3B3',  # 浅灰
        '#C2B0B0',  # 灰粉
        '#8C8C8C',  # 深灰
        '#6B8E23',  # 橄榄绿
        '#FF8C00',  # 暗橙
        '#4682B4'   # 钢蓝
]


def expand_color_list(colors, target_len):
    """将颜色列表通过插值扩充到目标长度"""
    if target_len <= len(colors):
        return colors[:target_len]
    
    # 将hex转RGB
    def hex_to_rgb(hex_color):
        hex_color = hex_color.lstrip('#')
        return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
        
    # RGB转hex
    def rgb_to_hex(rgb):
        return '#{:02X}{:02X}{:02X}'.format(int(rgb[0]), int(rgb[1]), int(rgb[2]))
    
    rgb_colors = [hex_to_rgb(c) for c in colors]
    rgb_colors = np.array(rgb_colors)
    
    # 在颜色环上均匀插值
    indices = np.linspace(0, len(rgb_colors) - 1, target_len)
    new_colors = []
    
    for i in indices:
        idx_low = int(np.floor(i))
        idx_high = int(np.ceil(i))
        if idx_low == idx_high:
            new_colors.append(rgb_colors[idx_low])
        else:
            # 线性插值
            frac = i - idx_low
            interp_color = (1 - frac) * rgb_colors[idx_low] + frac * rgb_colors[idx_high]
            new_colors.append(interp_color)
    
    # 添加轻微扰动避免颜色过于相似（可选，增强区分度）
    np.random.seed(42)  # 固定随机种子保证可重复性
    for i in range(len(new_colors)):
        # 轻微调整亮度/饱和度，幅度<5%
        noise = np.random.uniform(-0.05, 0.05, 3)
        new_colors[i] = np.clip(new_colors[i] + new_colors[i] * noise, 0, 255)
    
    return [rgb_to_hex(c) for c in new_colors]

# 扩充到需要的不同的颜色个数
Nat_color_list = expand_color_list(Nat_color_list, len(adata.obs['leiden'].unique()))



In [ ]:
######## 画下面的图之前先做下抽样，避免这么多细胞的点都挤在一起
adata_plot = sc.pp.subsample(
    adata,
    n_obs=300000,
    copy=True,
    random_state=0
)


In [ ]:
###################### 画 Fibroblast/Endothelial atlas 的 UMAP 图的 code, 以及后面 Fibroblast 中所有Celltype的配色都可以参考这个配色
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
# 1. 基础检查
required_cols = ['leiden', 'Celltype_general', 'Celltype']
for col in required_cols:
    if col not in adata.obs.columns:
        raise ValueError(f"adata.obs 中缺少必要列: {col}")

# 确保 leiden 是字符串（Scanpy 推荐）
adata.obs['leiden'] = adata.obs['leiden'].astype(str)
adata.obs['Celltype'] = adata.obs['Celltype'].astype(str)
# 根据adata.obs['leiden'] 产生 leiden color list, 参考Nature系列的配色

# ================================
# 2. 构建 Leiden → 颜色映射
# ================================
leiden_categories = sorted(
    adata.obs['leiden'].unique(),
    key=lambda x: int(x)
)
if len(leiden_categories) != len(Nat_color_list):
    raise ValueError("Nat_color_list 数量必须等于 Leiden cluster 数量")

leiden_color_dict = dict(zip(leiden_categories, Nat_color_list))
# 告诉 scanpy 使用这套颜色
adata.uns['leiden_colors'] = [
    leiden_color_dict[c] for c in leiden_categories
]
# ================================
# 3. 构建 Celltype_C 列
# ================================

adata.obs['Celltype_C'] = (
    adata.obs['Celltype'] +
    "_C" +
    adata.obs['leiden']
)
# ================================
# 4. 构建 Celltype_C → 颜色映射（来自 Leiden）
# ================================
celltypeC_color_dict = {}
for leiden_id in leiden_categories:
    mask = adata.obs['leiden'] == leiden_id
    celltypeC_values = adata.obs.loc[mask, 'Celltype_C'].unique()
    for val in celltypeC_values:
        celltypeC_color_dict[val] = leiden_color_dict[leiden_id]

# ================================
# 5. 开始绘图
# ================================
plt.rcParams['figure.figsize'] = (13, 6)
fig, ax = plt.subplots()
# 关键：指定 palette=Nat_color_list
sc.pl.umap(
    adata,
    color='leiden',
    palette=Nat_color_list,
    ax=ax,
    show=False,
    legend_loc='on data',   # 不用scanpy默认legend
    s=10,
    alpha=0.8
)
ax.set_title('Fibroblast Atlas', fontsize=16, fontweight='bold', pad=20)
# ================================
# 6. 去边框
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
# ----------------------
# 5. 添加带箭头的轴标签
# ----------------------
# 计算箭头长度（基于UMAP坐标范围，确保箭头比例协调）
x_range = ax.get_xlim()
y_range = ax.get_ylim()
arrow_length_x = (x_range[1] - x_range[0]) * 0.1  # 占X轴范围10%
arrow_length_y = (y_range[1] - y_range[0]) * 0.1  # 占Y轴范围10%
# 添加UMAP_1箭头（x轴0点向右）
ax.annotate(
    'UMAP_1',
    xy=(0, 0),  # 箭头起点（坐标轴0点）
    xytext=(arrow_length_x, 0),  # 箭头终点
    arrowprops=dict(
        arrowstyle='<-',  # 箭头样式
        color='black',    # 箭头颜色
        lw=1.5            # 箭头线宽
    ),
    fontsize=13,
    ha='left',  # 文本对齐方式
    va='center'
)
# 添加UMAP_2箭头（y轴0点向上，竖直文本）
ax.annotate(
    'UMAP_2',
    xy=(0, 0),  # 箭头起点（坐标轴0点）
    xytext=(0, arrow_length_y),  # 箭头终点
    arrowprops=dict(
        arrowstyle='<-',  # 箭头样式
        color='black',    # 箭头颜色
        lw=1.5            # 箭头线宽
    ),
    fontsize=13,
    ha='center',  # 文本对齐方式
    va='bottom',
    rotation=90   # 文本竖直排布
)

# ================================
# 8. 构建分组 legend（按 Celltype_general）
# ================================
general_groups = ['qFibro', 'Myfibro', 'iFibro', 'eFibro']
col1_generals = [ 'Myfibro', 'iFibro']
col2_generals = ['eFibro','qFibro']
def build_legend_handles(generals):
    handles = []
    labels = []
    for g in generals:
        subset = adata.obs[adata.obs['Celltype_general'] == g]
        unique_celltypeC = sorted(subset['Celltype_C'].unique())
        # 添加 general 标题（黑色）
        handles.append(mpatches.Patch(color='white', label=g))
        labels.append(g)
        # 添加每个 Celltype_C
        for ct in unique_celltypeC:
            handles.append(
                mpatches.Patch(
                    color=celltypeC_color_dict[ct],
                    label=ct
                )
            )
            labels.append("  " + ct)  
    return handles, labels


# 左 legend
handles1, labels1 = build_legend_handles(col1_generals)
legend1 = ax.legend(
    handles1,
    labels1,
    bbox_to_anchor=(1.02, 1.1),
    loc='upper left',
    frameon=False,
    fontsize=9
)
ax.add_artist(legend1)
# 右 legend
handles2, labels2 = build_legend_handles(col2_generals)
legend2 = ax.legend(
    handles2,
    labels2,
    bbox_to_anchor=(1.4, 1.1),
    loc='upper left',
    frameon=False,
    fontsize=9
)
# ----------------------
# 8. 计算每个 Celltype_general 对应的UMAP坐标边界（用于画圈）
# ----------------------
def get_cluster_bounds(adata, group_col):
    """计算每个分组的UMAP坐标边界（中心+宽高）"""
    bounds = {}
    for group in adata.obs[group_col].unique():
        # 获取该分组的UMAP坐标
        umap_coords = adata.obsm['X_umap'][adata.obs[group_col] == group]
        if len(umap_coords) == 0:
            continue
        # 计算中心坐标
        center_x = np.mean(umap_coords[:, 0])
        center_y = np.mean(umap_coords[:, 1])
        # 计算边界（用于确定圆圈大小）
        min_x, max_x = np.min(umap_coords[:, 0]), np.max(umap_coords[:, 0])
        min_y, max_y = np.min(umap_coords[:, 1]), np.max(umap_coords[:, 1])
        # 计算宽度和高度（扩大1.2倍，让圈更宽松）
        width = (max_x - min_x) * 1.2
        height = (max_y - min_y) * 1.2
        bounds[group] = {
            'center': (center_x, center_y),
            'width': width,
            'height': height
        }
    return bounds

# 计算每个Celltype_general的UMAP边界
cluster_bounds = get_cluster_bounds(adata, 'Celltype_general')

# ----------------------
# 9. 为每个Celltype_general添加弯曲虚线圆圈和文本标签
general_colors = {
    'qFibro': '#ff7f0e',    # 橙色
    'Myfibro': '#d62728',   # 红色
    'iFibro': '#2ca02c' ,   # 绿色
    'eFibro': '#1f77b4'    # 蓝色
}
# ----------------------
for general_type, color in general_colors.items():
    if general_type not in cluster_bounds:
        continue
        
    # 获取该分组的边界信息
    bounds = cluster_bounds[general_type]
    center_x, center_y = bounds['center']
    width = bounds['width']
    height = bounds['height']
    # 创建弯曲的椭圆虚线框（模拟弯曲圈）
    ellipse = mpatches.Ellipse(
        (center_x, center_y),
        width=width,
        height=height,
        edgecolor=color,
        facecolor='none',
        linestyle='--',  # 虚线
        linewidth=2,     # 线宽
        alpha=0.8        # 透明度
    )
    ax.add_patch(ellipse)
    # 添加文本标签（在圈的右侧/上方，避免遮挡）
    # 计算标签位置（圈的右侧偏移一点）
    label_x = center_x + width/2 + (x_range[1]-x_range[0])*0.02
    label_y = center_y
    # 添加文本
    ax.text(
        label_x, label_y,
        general_type,
        fontsize=12,
        fontweight='bold',
        color=color,
        ha='left',
        va='center',
        bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor=color, alpha=0.7)
    )

# 10. 保存
plt.tight_layout()
fig_dir='external_inputs/OnlyFibro'
plt.savefig(
    fig_dir+"Fibroblast_umap_leiden_Celltype.pdf",
    dpi=300,
    bbox_inches='tight',
    format='pdf'
)
#plt.savefig(fig_dir+"/Fibroblast_umap_leiden_Celltype.png", dpi=300, bbox_inches='tight',pad_inches=0.1,transparent=False)
plt.show()


In [ ]:

############### 改成 4 个subtype后，产王老师说的那种 boxplot, 柱子窄一点的 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
# 设置Nature/Cell期刊风格的参数
plt.style.use('default')
rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})
# 1. 提取数据框并添加Celltype信息
adata_meta = adata.obs[['leiden','Celltype_general','Celltype', 'Myfibro', 'eFibro', 'iFibro', 'qFibro', 'Myfibro1', 'eFibro1','iFibro1', 'qFibro1']].copy()
adata_meta['Celltype'] = adata_meta['Celltype'].astype(str) + '_C' + adata_meta['leiden'].astype(str)  
# 定义leiden到Celltype的映射字典
# 确保leiden列是字符串类型，方便处理
adata_meta['leiden'] = adata_meta['leiden'].astype(str)
# 3. 按照leiden分组计算平均值
# 注意：这里修正了列名大小写问题
feature_columns = ['Myfibro', 'eFibro', 'iFibro', 'qFibro']   #  'Myfibro1', 'eFibro1','iFibro1', 'qFibro1'
# 检查列名是否存在，确保大小写匹配
available_columns = [col for col in feature_columns if col in adata_meta.columns]
grouped_mean = adata_meta.groupby(['Celltype'])[available_columns].mean().reset_index()
grouped_mean['leiden'] = grouped_mean['Celltype'].str.split('_C').str[-1].astype(int)
grouped_mean = grouped_mean.sort_values(['Celltype','leiden'])
print(grouped_mean[['leiden', 'Celltype']].to_string(index=False))
# 5. 可视化设置
n_leiden = len(grouped_mean)
print(f"\nLeiden群数: {n_leiden}")
# 设置子图布局：4行10列
n_rows =  5
n_cols =  11
n_subplots = n_rows * n_cols
# 如果leiden数量少于子图数量，调整布局
if n_leiden < n_subplots:
    n_cols = int(np.ceil(n_leiden / n_rows))
    n_subplots = n_rows * n_cols

colors = plt.cm.Set2(np.linspace(0, 1, len(available_columns)))
# 6. 创建图形
fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.5*n_cols, 2*n_rows))
axes = axes.flatten()
# 设置图形标题
fig.suptitle('Signature score of subFibro markers',  # Signature score of subFibro(long list)
             fontsize=12, fontweight='bold', y=1.02)
# 7. 绘制每个leiden的柱状图（添加数值标签）
for i, row in enumerate(grouped_mean.itertuples()):
    if i >= len(axes):
        break
    
    leiden = row.leiden
    celltype = row.Celltype
    ax = axes[i]
    # 提取当前leiden的数据
    leiden_data = grouped_mean.loc[grouped_mean['leiden'] == leiden, available_columns].iloc[0]
    # 创建柱状图
    x_pos = np.arange(len(available_columns))
    bars = ax.bar(x_pos, leiden_data.values, color=colors, edgecolor='black', linewidth=0.5)
    # 添加数值标签到每个柱子顶部
    for bar_idx, (bar, value) in enumerate(zip(bars, leiden_data.values)):
        # 确定标签位置
        height = bar.get_height()
        # 如果值太小，将标签放在柱子内部顶部
        if height < grouped_mean[available_columns].values.max() * 0.05:
            # 小值：标签放在柱子内部，白色字体
            text_y = height * 0.8  # 放在柱子高度的80%位置
            color = 'black' # 小的值也还是 black
            va = 'bottom'
        else:
            # 正常值：标签放在柱子顶部上方
            text_y = height + grouped_mean[available_columns].values.max() * 0.01  # 稍微高于柱子
            color = 'black'
            va = 'bottom'
        
        # 格式化数值显示（根据数值大小选择格式）
        if abs(value) < 0.001:
            text = f'{value:.1e}'
        elif abs(value) < 0.01:
            text = f'{value:.4f}'
        elif abs(value) < 0.1:
            text = f'{value:.3f}'
        else:
            text = f'{value:.2f}'
        
        # 添加文本标签
        ax.text(
            bar_idx, text_y, text,
            ha='center', va=va,
            fontsize=6,  # 使用较小字体
            color=color,
            fontweight='bold' if color == 'white' else 'normal',
            rotation=0
        )
    # 设置子图标题（包含leiden和Celltype信息）
    # 缩短Celltype显示，只显示主要部分
    if len(celltype) > 20:
        # 如果Celltype太长，只显示前部分
        short_celltype = celltype # celltype[:17] + "..."
    else:
        short_celltype = celltype
    
    title_text =f'{short_celltype}'  # f'L{leiden}\n{short_celltype}'
    ax.set_title(title_text, fontsize=6, fontweight='bold', pad=3)
    # 隐藏x轴刻度
    ax.set_xticks([])
    # 设置y轴标签
    if i % n_cols == 0:  # 每行最左边的图显示y轴标签
        ax.set_ylabel('Signature Score', fontsize=8)
    
    # 设置y轴范围，为顶部标签留出空间
    y_max = grouped_mean[available_columns].values.max() * 1.15  # 增加15%的空间用于标签
    ax.set_ylim([0, y_max])
    # 添加网格线（仅y方向，更简洁）
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    # 移除顶部和右侧边框
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # 根据Celltype前缀添加背景色区分
    celltype_prefix =  celltype.split('_')[0]     # get_celltype_prefix(celltype) , 这个函数不在了
    if celltype_prefix == 'iFibro':  # iFibro
        ax.set_facecolor('#FFF5E6')  # 浅橙色背景
    elif celltype_prefix == 'eFibro':  # eFibro
        ax.set_facecolor('#E6F7FF')  # 浅蓝色背景
    elif celltype_prefix == 'Myfibro':  # Myfibro
        ax.set_facecolor('#F0E6FF')  # 浅紫色背景
    elif celltype_prefix == 'qFibro':  # qFibro
        ax.set_facecolor('#E6FFE6')  # 浅绿色背景

# 8. 隐藏多余的子图
for i in range(len(grouped_mean), len(axes)):
    axes[i].axis('off')

# 9. 创建统一的图例
# 使用简化版特征名称作为图例标签
feature_short_names = []
for col in available_columns:
    if 'Myfibro1' in col:
        feature_short_names.append('Myfibro')
        # feature_short_names.append('Myfibro_l')
    elif 'iFibro1' in col:
        feature_short_names.append('iFibro')
    elif 'eFibro1' in col:
        feature_short_names.append('eFibro')
    elif 'qFibro1' in col:
        feature_short_names.append('qFibro')
    elif 'Myfibro' in col:
        feature_short_names.append('Myfibro')
    elif 'iFibro' in col:
        feature_short_names.append('iFibro')
    elif 'eFibro' in col:
        feature_short_names.append('eFibro')
    elif 'qFibro' in col:
        feature_short_names.append('qFibro')
    else:
        feature_short_names.append(col.replace('_4raw', ''))

# 创建自定义的图例句柄
legend_handles = [plt.Rectangle((0,0),1,1, color=colors[j], edgecolor='black', linewidth=0.5) 
                  for j in range(len(available_columns))]
# 创建Celltype分组图例
celltype_prefix_colors = {
    1: ('Myfibro', '#F0E6FF'),
    2: ('eFibro', '#E6F7FF'),
    3: ('iFibro', '#FFF5E6'),
    4: ('qFibro', '#E6FFE6')
}
# 创建Celltype图例句柄
celltype_handles = []
celltype_labels = []
for prefix in sorted(celltype_prefix_colors.keys()):
    label, color = celltype_prefix_colors[prefix]
    handle = plt.Rectangle((0,0),1,1, color=color, edgecolor='gray', linewidth=0.5, alpha=0.7)
    celltype_handles.append(handle)
    celltype_labels.append(label)

# Subtypes图例（左下区域）
fig.legend(legend_handles, feature_short_names, 
           loc='lower center', 
           bbox_to_anchor=(0.25, 0.01),  # 底部左侧
           ncol=len(available_columns),  # 水平排列
           fontsize=8,
           title='marker_list',
           title_fontsize=9,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

# Celltype Groups图例（右下区域）
fig.legend(celltype_handles, celltype_labels,
           loc='lower center',
           bbox_to_anchor=(0.75, 0.01),  # 底部右侧
           ncol=4,  # 水平排列
           title='Celltype',
           title_fontsize=9,
           fontsize=8,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

plt.tight_layout()
plt.subplots_adjust(top=0.95, bottom=0.1,left=0.05, right=0.9, hspace=0.4, wspace=0.3)
# 12. 保存图形
fig_path = 'external_inputs/OnlyFibro'
# plt.savefig(
#     fig_path+"leiden_subtype_Short_scores.pdf",
#     dpi=300,
#     bbox_inches='tight',
#     format='pdf')
fig.savefig(fig_path+'leiden_subtype_Short_scores.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
##### 更换 raw 数据重新算 signature score, 不能用 PCA的数据，需要使用原始的 pickle 文件，但 SS很奇怪
Fib_pca = sc.read_h5ad('/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_PCA.h5ad')
adata_cellid = adata.obs_names
existing_cells = [cell for cell in adata_cellid if cell in Fib_pca.obs_names]
missing_cells = [cell for cell in adata_cellid if cell not in Fib_pca.obs_names]
Fib_pca = Fib_pca[existing_cells, :]  # 需确保 len(existing_cells) = len(adata_cellid) ; 不然更换表达矩阵会出问题


In [ ]:
#### 挑选 Age的数据，后面分成 healthy & disease 来训练数据
adata_age  = adata[~np.array(list(map(lambda x: isstr(x), adata.obs['age'])))]
adata_age.obs['age'] = adata_age.obs['age'].astype(float) 
adata_age.obs['Age'] = adata_age.obs['age']
adata_age.write('/data/project/chenfeiwang/ytwang/Fibro_human_FibDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyFib_age.h5ad')
